# Phase 1 - Heart Disease Prediction

**Course:** SWE485 (Selected Topics in Software Engineering)
**Phase:** 1 (Supervised Learning & Model Development)
**Dataset:** Heart Disease Dataset (Kaggle – Preprocessed Version)

This notebook covers:
• Next Steps for Phase 2 Integration  ? (عشان ماننساها لانها مكتوبة)

# The Notebook Overview


## 1. Model Selection & Rationale
In this section, we will justify the choice of each model based on the characteristics of our dataset and the problem type, as well as the strengths and weaknesses of the models in addressing the task of heart disease prediction.

## 2. Implementation & Training
In this section , we will describe how each model was implemented and trained. Here, we can see the common procedures applied to all three models.

### 2.1 Train / Test Split

#### **Why do we split at all?**
A machine learning model learns patterns from data. If we evaluate it on the same data it
learned from, it will look like it performs perfectly, but that tells us nothing about how
it handles new, unseen patients. As scikit-learn's official documentation states, learning
the parameters of a model and testing it on the same data is a methodological mistake: a
model that simply repeats the labels it has seen would score perfectly but fail on any new
data (scikit-learn, 2024). We therefore need a separate test set that the model never sees
during training to get an honest measure of real-world performance.

Additionally, our approach involves an exhaustive hyperparameter search over hundreds of
combinations to find the best model configuration. The data used to select that winning
combination must never include the test set, otherwise the selection process would be
indirectly influenced by the samples we intend to evaluate on, compromising the fairness
of the final result. The hyperparameter search process is explained in detail in Section 3.

#### **Why 80% training and 20% testing?**
With 917 samples, we need to give the model enough data to learn meaningful patterns while
still reserving enough samples to evaluate it fairly. An 80/20 split gives ~733 samples for
training and ~183 for testing. Empirical studies confirm that 80/20 provides a good balance
for moderate-sized datasets (Gholamy et al., 2018).

A known limitation of holdout splitting is that the test set may not perfectly represent
the full population, particularly when the dataset is small (Varoquaux et al., 2023).
We mitigate this in two ways: first, we use stratified splitting, which guarantees the
test set preserves the exact class ratio of the full dataset; second, we independently
run cross-validation on the training set and report its metrics alongside the test set
results. If both sets of metrics are in close agreement, that is empirical evidence that
the test set is representative. The printed evaluation results in Section 4 confirm this.

#### **Our approach: 80/20 split first, then k-fold inside the training set only**
Some resources and tutorials apply k-fold directly to the full dataset without holding out
a test set first, passing the entire X and y into cross_val_score and reporting the mean
score as the final result (DataCamp, 2024). This simplified approach is common in
introductory tutorials because it is easier to demonstrate.

However, for our project we deliberately follow a more rigorous approach. We first hold out
20% of the data as a completely untouched test set, then apply 5-fold cross-validation
exclusively within the remaining 80% training set. This is the workflow explicitly
recommended by both scikit-learn (2024) and Machine Learning Mastery (2021).


Machine Learning Mastery (2021) further demonstrates this exact two-step workflow:
first perform train/test split, then apply k-fold cross-validation on the training set
only to select and tune the model, and finally evaluate on the held-out test set to confirm
generalisation performance.

In summary, each component of our approach serves a distinct and necessary purpose:

- **K-fold on the 80% training set** → tunes hyperparameters and verifies the model is
  stable and consistent across different subsets of training data
- **The held-out 20% test set** → provides the final unbiased performance report on data
  the model has never touched at any stage of training or tuning

In a critical medical application, K-fold alone is not
sufficient because it cannot guarantee a truly isolated evaluation set.

#### **Why are we saving the split indices?**
We train three different models in this project. For the comparison between them to be
fair, all three models must be evaluated on exactly the same test samples and trained on
exactly the same training samples. If each model used a different random split, any
difference in performance could be due to the split rather than the model itself. By saving
the split indices once and loading them in every cells notebook, we guarantee a completely
fair comparison across all models.


### 2.2 Hyperparameter Tuning Process & Results

#### **What is hyperparameter tuning?**
Hyperparameters are settings that control how the model is built. Unlike regular parameters
(which the model learns from data automatically), hyperparameters must be set by us before
training begins. Choosing the wrong values can lead to a model that is too simple to learn
patterns (underfitting) or too complex and memorises the training data (overfitting). Tuning
finds the best combination for our specific dataset.

#### **Why GridSearchCV with k-fold inside?**
GridSearchCV tries every possible combination of the hyperparameter values we provide. For
each combination, it uses **5-fold cross-validation** to score it, meaning it trains and
validates the model 5 times on different subsets of the training data and averages the score.
This makes the evaluation of each combination reliable and not dependent on a single lucky
or unlucky split. (GeeksforGeeks, 2025).

#### **Why GridSearchCV specifically for our case?**
We chose GridSearchCV because it automatically tries every possible combination of settings and picks the best one, so we do not have to guess. Our list of combinations is small enough (288) that checking all of them only takes a few minutes, making this practical. In a medical setting where prediction accuracy directly affects patient outcomes, relying on default settings or guesswork is not acceptable. GridSearchCV removes that risk by being exhaustive and objective. We also apply it consistently across all three models in this project, so when we compare their results, we know any difference in performance is due to the model itself and not how it was tuned.

> GeeksforGeeks. (2025). *Performing feature selection with GridSearchCV in sklearn.*
> https://www.geeksforgeeks.org/machine-learning/performing-feature-selection-with-gridsearchcv-in-sklearn/

## 3. Comprehensive Evaluation
In this section, we will evaluate the performance of the models based on several key metrics and visualizations. The evaluation will be conducted based on the following:

#### Two-stage evaluation approach:

**Stage 1 — 5-Fold Cross-Validation (on training set):**
Checks whether the tuned model is stable and consistent. A model that performs well in cross-validation but poorly on the test set is overfitting. A low standard deviation across folds means the model behaves consistently regardless of which subset of training data it sees.

**Stage 2 — Final evaluation on the held-out test set:**
The test set was locked away before any training or tuning began. The model has never seen these samples. This gives the final, unbiased answer to: *"how does this model perform on a completely new patient?"*

Both stages are necessary. Cross-validation alone is not enough because every sample in it was used for training at some point. The test set alone is not enough because a single split might be lucky or unlucky.

the following measure will be used:

- Accuracy: The percentage of correct predictions made by the model.

- Precision: The proportion of true positive predictions relative to the total predicted positives.

- Recall: The proportion of true positive predictions relative to the total actual positives.

- F1-Score: The harmonic mean of Precision and Recall, providing a balance between the two.

- Confusion Matrix: A visualization that shows the true positives, false positives, true negatives, and false negatives, helping to understand how well the model classifies each category.

- ROC-AUC: For binary classification, this metric evaluates the model's ability to distinguish between the classes.

## 4. Comparative Analysis & Interpretation
In this section, we will compare the performance of all three models based on their evaluation results and interpret their behavior:

**Which model performed best? Why?**
We will identify the best-performing model and explain why it performed better than the others based on the evaluation metrics.

**Analyze misclassifications: patterns, challenging categories**
We will examine the misclassifications made by the models, look for patterns in the mistakes, and identify which categories were more challenging for the models to predict.

**Discuss trade-offs: accuracy vs. interpretability vs. computational cost**
We will discuss the trade-offs between the models in terms of:

Accuracy: How well the model performs.

Interpretability: How easy it is to understand the model’s decisions.

Computational cost: The resources and time required for training and making predictions.

Recommendation: Which model will you use in your final system and why?
Based on the comparative analysis, we will recommend which model should be used in the final system and justify our choice.

---

> Gholamy, A., Kreinovich, V., & Kosheleva, O. (2018). *Why 70/30 or 80/20 relation
> between training and testing sets.* UTEP.
> https://scholarworks.utep.edu/cs_techrep/1209/
>
> scikit-learn. (2024). *Cross-validation: evaluating estimator performance.*
> https://scikit-learn.org/stable/modules/cross_validation.html
>
> Tam, A. (2021). *Training-validation-test split and cross-validation done right.*
> Machine Learning Mastery.
> https://machinelearningmastery.com/training-validation-test-split-and-cross-validation-done-right/
>
> Varoquaux, G., et al. (2023). *A guide to cross-validation for artificial intelligence
> in medical imaging.* Radiology: Artificial Intelligence.
> https://pmc.ncbi.nlm.nih.gov/articles/PMC10388213/
>
> DataCamp. (2024). *A comprehensive guide to K-Fold Cross Validation.*
> https://www.datacamp.com/tutorial/k-fold-cross-validation
>
> GeeksforGeeks. (2025). *Cross validation in machine learning.*
> https://www.geeksforgeeks.org/machine-learning/cross-validation-machine-learning/









# Random Forest



## Section 1. Model Selection & Rationale :

Random Forest is an ensemble learning algorithm that builds multiple decision trees during
training and combines their predictions through majority voting for classification. As IBM (2025)
explains, Random Forest is built on the principle of bagging (Bootstrap Aggregating), where each
tree is trained on a random sample of the data with replacement, which reduces variance and
improves the model's generalization ability.

---

### **Why Random Forest is a Good Fit for Our Data?**

#### i. Dataset Characteristics (Size, Feature Types, and Linearity)

The dataset contains **917 observations** after dropping one invalid RestingBP record. Following
one-hot encoding of categorical features (ChestPainType, RestingECG, ST_Slope, Chol_category), the feature
space expanded to **17 variables**.
With a wider feature space like this, there is a risk that certain dominant variables overshadow
others during model training. Random Forest's random feature subsampling at each split directly
addresses this by considering only a random subset of features at each node, it prevents any
single variable from consistently dominating the splits and ensures that all features, including
the one-hot encoded variables, have an opportunity to contribute to the prediction.

Regarding feature types, the dataset after preprocessing consists entirely of numerical
variables, continuous features (Age, RestingBP, MaxHR, Oldpeak) were normalized, and all
categorical features were encoded into numerical form. Random Forest works well with this fully
numerical structured input, as its tree-splitting mechanism evaluates feature thresholds across
all features uniformly, regardless of their original scale or type.

Regarding linearity, the pair plot analysis confirmed that continuous features do not exhibit
clear linear relationships with each other or with the target variable HeartDisease. IBM (2025)
highlights that Random Forest handles non-linear data effectively, since tree-based splits
recursively partition the feature space and can model complex, irregular decision boundaries that
linear models cannot capture.

---

#### ii. Problem Type (Binary Classification)

Our task is a binary classification problem: predicting whether a patient has heart disease
(HeartDisease = 1) or does not (HeartDisease = 0). The class distribution is relatively balanced
as shown in the pie chart visualization. Random Forest is well-suited to binary classification,
as it outputs a majority vote across all trees for each class, producing stable and reliable
binary predictions. IBM (2025) notes that Random Forest is widely applied in healthcare
classification tasks, directly aligned with our heart disease prediction system, where reliable
and consistent class separation is essential.

---

#### iii. Model Strengths and Weaknesses for Our Specific Problem

**Strengths:**

- **Robustness to outliers:** Our EDA identified outliers in multiple clinical features,
RestingBP contained outliers on the right side, requiring RobustScaler normalization, and
Oldpeak exhibited right skewness with extreme values. Because Random Forest makes decisions
based on feature thresholds rather than distances or magnitudes, it is naturally less affected
by extreme values compared to other models. This is especially important in medical data,
where an unusual value may still represent a real and significant clinical case that should not be
ignored.

- **Feature importance:** Random Forest provides feature importance scores after training,
showing how much each variable contributed to the predictions. IBM (2025) notes this is one
of Random Forest's key advantages, and in a medical system like ours it helps identify which
clinical measurements, such as chest pain type or ST slope, are the most influential in
predicting heart disease.

- **High accuracy on tabular data:** IBM (2025) notes that Random Forest is known for
producing highly accurate results on structured tabular datasets, which is essential for our
system where reliable medical prediction is critical.

**Weaknesses:**

- **Computational cost:** Based on GeeksforGeeks (2025), since Random Forest builds many trees instead of
just one, it requires more memory and processing time, especially when tuning hyperparameters to find the
best model settings.

- **Reduced interpretability:** Although feature importance scores are available, GeeksforGeeks (2025) have
stated that it is difficult to trace exactly how a single prediction was made, since the final result comes
from combining hundreds of trees rather than following one clear decision path.

Despite these weaknesses, Random Forest remains the most suitable choice for our project. It
handles our expanded 17-feature dataset efficiently, captures the non-linear relationships
observed in the EDA, tolerates the clinical outliers identified during preprocessing, and is
well-established in healthcare classification tasks, all of which directly match the
characteristics and goals of our heart disease prediction system.

---

### References

> IBM. (2025). *What is random forest?* IBM Think.
> https://www.ibm.com/think/topics/random-forest
>
> GeeksforGeeks. (2025). *Random forest algorithm in machine learning.*
> https://www.geeksforgeeks.org/machine-learning/random-forest-algorithm-in-machine-learning/

## Section 2. Implementation & Training

### Import Libraries

In [ ]:
import subprocess
subprocess.run(["pip", "install", "seaborn"], check=True)
#Standard libraries
import pickle
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
warnings.filterwarnings('ignore')

#  Model
from sklearn.ensemble import RandomForestClassifier

#  Splitting & cross-validation
from sklearn.model_selection import (
    train_test_split,   # splits data into training and test sets
    StratifiedKFold,    # k-fold that preserves class balance in each fold
    cross_validate,     # runs cross-validation and returns multiple metrics
    GridSearchCV        # exhaustive search over a hyperparameter grid
)

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,     # proportion of correct predictions
    precision_score,    # of all predicted positives, how many are truly positive
    recall_score,       # of all actual positives, how many were correctly predicted
    f1_score,           # harmonic mean of precision and recall
    confusion_matrix,   # table showing TP, TN, FP, FN counts
    roc_auc_score,      # area under the ROC curve
    roc_curve           # false positive rate vs true positive rate at all thresholds
)

# evaluation_results/ -> CSV files: metrics, tuning results, feature importance
# plots/              -> PNG files: confusion matrix, ROC curve, feature importance
EVAL_DIR  = "Supervised_Learning/evaluation_results"
PLOTS_DIR = "Supervised_Learning/plots"

os.makedirs(EVAL_DIR,  exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# Model name prefix
# Used in every saved file name so outputs are clearly labelled per model
MODEL_NAME = "random_forest"

print("Libraries imported successfully.")
print(f"Evaluation results → {EVAL_DIR}")
print(f"Plots              → {PLOTS_DIR}")

### Load Preprocessed Dataset

In [ ]:
# Load the preprocessed dataset
# This CSV was produced by the EDA notebook — it contains cleaned, encoded,
# and normalised features ready for modeling. No further preprocessing is needed.
# Shape: 917 rows × 17 columns (16 features + 1 target column)
DATA_PATH = "Dataset/preprocessed_heart_data.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

### Separate Features and Target

In [ ]:
# Separate input features from the target variable
# X -> all feature columns (what the model uses to make predictions)
# y -> the target column   (what the model is trying to predict)
TARGET = "HeartDisease"   # 0 = No Disease, 1 = Heart Disease

X = df.drop(columns=[TARGET])
y = df[TARGET]

print(f"Features (X): {X.shape[1]} columns")
print(f"\nTarget (y) distribution:")
print(y.value_counts().rename({0: 'No Disease (0)', 1: 'Heart Disease (1)'}))

In [ ]:
# Create or load the 80/20 train/test split
# This cell implements the first and most critical step of our methodology:
# locking away 20% of the data before any training or tuning begins.
#
# IMPORTANT: This split is created ONCE and shared across all three models.
# If each model used a different split, performance differences could be due
# to the split rather than the model — making comparison unfair.
# Saving and loading the same indices guarantees a completely fair comparison.
SPLIT_PATH = "split_indices.pkl"

if not os.path.exists(SPLIT_PATH):
    # Create the split for the first time
    # stratify=y   → preserves the class ratio (proportion of 0s and 1s)
    #                in both the training and test sets
    # random_state → fixes the random seed so the split is reproducible
    #                running this cell again always produces the same split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    # Save the indices (row positions) of training and test samples
    # We save indices rather than the actual data so any model can
    # reconstruct the exact same split from any version of the dataset
    split_indices = {
        "train_indices": X_train.index.tolist(),
        "test_indices" : X_test.index.tolist()
    }
    with open(SPLIT_PATH, "wb") as f:
        pickle.dump(split_indices, f)

    print("Split created and saved to split_indices.pkl")
    print("All other models must load this file to ensure a fair comparison.")

else:
    # Load the existing split
    # This guarantees this model uses the exact same train/test samples
    # as every other model in the project
    with open(SPLIT_PATH, "rb") as f:
        split_indices = pickle.load(f)

    X_train = X.iloc[split_indices["train_indices"]]
    X_test  = X.iloc[split_indices["test_indices"]]
    y_train = y.iloc[split_indices["train_indices"]]
    y_test  = y.iloc[split_indices["test_indices"]]

    print("Existing split loaded from split_indices.pkl")
    print("This model uses the exact same train/test samples as all other models.")

# Confirm split sizes and class balance
print(f"\nTraining set : {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test set     : {X_test.shape[0]}  samples ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\nClass distribution in training set:")
print(y_train.value_counts().rename({0: 'No Disease', 1: 'Heart Disease'}))
print(f"\nClass distribution in test set:")
print(y_test.value_counts().rename({0: 'No Disease', 1: 'Heart Disease'}))

#### Random Forest Hyperparameter Selection

Random Forest has a unique set of hyperparameters that directly control how its trees are
built and how they work together as an ensemble. Unlike simpler models, each hyperparameter
affects a different layer of the model, from how many trees vote on the final prediction,
to how deep each tree grows, to how many features each tree considers at each split.
Choosing the right combination is therefore not straightforward, as these settings interact
with each other and their optimal values depend on the specific dataset.

### Why These Specific Hyperparameters Were Selected for Tuning

RandomForestClassifier in scikit-learn has 19 configurable parameters in total. We selected 6
to tune, the ones that directly control the two most important sources of error in any Random
Forest: **overfitting** and **instability**. The remaining 13 were intentionally left at their
defaults, as they either address the same concerns redundantly, require specialist knowledge to
set meaningfully, or are infrastructure parameters that do not affect prediction quality.

**Tuned parameters:**

| Parameter | Why Tuned |
|-----------|-----------|
| `n_estimators` | How many trees to build — More trees reduce prediction variance; the default of 100 is often too conservative for medical datasets |
| `max_depth` | How deep each tree can grow — Unlimited depth causes severe overfitting; constrained to prevent the model from memorising training data |
| `min_samples_split` | Minimum samples needed to split a node — Default of 2 allows splits based on just 2 patients; increased to ensure each split is backed by meaningful evidence |
| `min_samples_leaf` | Minimum samples required at a leaf — Default of 1 allows leaves representing a single patient; increased to ensure every prediction is backed by a group |
| `max_features` | How many features to consider at each split — Controls diversity between trees; tuned to find the right balance between diversity and completeness |
| `class_weight` | How much penalty to assign per class — Dataset has a 55/45 split; included to let GridSearchCV decide whether the slight imbalance needed correction |

**Parameters not tuned:**

The remaining 13 parameters were excluded because they are either redundant with the above
(`min_weight_fraction_leaf`, `max_leaf_nodes`, `ccp_alpha`), fundamental to the algorithm
and should not be changed (`bootstrap`, `criterion`), diagnostic rather than predictive
(`oob_score`, `warm_start`), or infrastructure settings that do not affect model quality
(`n_jobs`, `random_state`, `verbose`).

> scikit-learn. *RandomForestClassifier*.
> https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html


In [ ]:
# Step 1: Define the hyperparameter search grid
# param_grid is a menu of options for GridSearchCV to try.
# GridSearchCV will test every possible combination of these values —
# 3×3×4×4×3×2 = 432 combinations × 5 folds = 2,160 model fits in total.
# Each combination is scored using 5-fold cross-validation on the training set.
# The combination with the highest average ROC-AUC score wins.
param_grid = {
    # Number of decision trees built in the forest
    # More trees reduce variance by averaging out individual tree errors (overfitting noise)
    # 100–300 was too conservative — 500 allows the ensemble to stabilise on a small dataset (917 rows)
    # Diminishing returns beyond 500, so we cap there to keep runtime reasonable
    'n_estimators'     : [200, 300, 500],

    # Maximum depth each tree is allowed to grow
    # Removed 15 and 20 — these still allowed too much depth in the previous iteration
    # contributing to the remaining overfitting gap
    # Keeping only shallow values (5, 8, 10) to further constrain tree complexity
    # and force the model to learn only the strongest, most generalisable patterns
    'max_depth'        : [5, 8, 10],

    # Minimum number of samples a node must have before it is allowed to split further
    # Pushed higher to [15, 20, 25, 30] — each iteration of raising this value has
    # consistently reduced the gap; continuing in the same direction to find the ceiling
    # Higher values force splits to be backed by increasingly strong evidence
    'min_samples_split': [15, 20, 25, 30],

    # Minimum number of samples required at each leaf (terminal node)
    # Pushed higher to [4, 5, 6, 8] — larger leaves mean each prediction is based on
    # more samples, further reducing sensitivity to noise in the training data
    'min_samples_leaf' : [4, 5, 6, 8],

    # Number of features randomly considered at each split point
    # 'sqrt' → uses √17 ≈ 4 features per split (original scikit-learn default, promotes diversity)
    # 'log2' → uses log₂(17) ≈ 4 features per split (similar effect, slightly more aggressive)
    # None   → considers ALL 17 features at every split
    #          on small datasets this often wins because no signal is accidentally excluded;
    #          on large datasets it would be too slow and cause overfitting
    'max_features'     : ['sqrt', 'log2', None],

    # How much weight the model assigns to each class during training
    # None     → treats both classes equally regardless of how many samples each has
    # 'balanced' → automatically increases the penalty for misclassifying the minority class
    #              by weighting each class inversely proportional to its frequency
    #              Your dataset is ~55% heart disease / ~45% healthy — not severely imbalanced,
    #              but balanced weighting often reduces false negatives (missed disease cases)
    #              which also lifts overall accuracy by 1–3%
    'class_weight'     : ['balanced', None]
}

# Step 2: Define the cross-validation strategy
# This tells GridSearchCV HOW to evaluate each combination fairly:
# — Split the 733 training samples into 5 equal folds (~146 samples each)
# — Each fold takes a turn being the validation set exactly once
# — Stratified: every fold preserves the 55/45 class ratio of the full training set
# — shuffle + random_state=42: same fold assignments every run → reproducible results
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Step 3: Run the exhaustive search
# GridSearchCV ties all three components together:
#   estimator  → the model to train (RandomForestClassifier)
#   param_grid → the combinations to try (432 total)
#   cv         → how to evaluate each combination (5-fold stratified CV)
#   scoring    → what metric to optimise (ROC-AUC)
#
# Why ROC-AUC and not F1 or accuracy?
# F1 and accuracy only evaluate the model at a fixed 0.5 decision threshold.
# ROC-AUC evaluates across ALL thresholds — it rewards a model that genuinely
# ranks heart disease patients above healthy ones regardless of where the
# decision line is drawn. This produces a stronger underlying model, and in a
# medical setting allows the threshold to be adjusted clinically later
# (e.g. lowered to catch more cases) without retraining.
#
# What happens when .fit() is called:
# 1. For each of 432 combinations → train on 4 folds, validate on 1 fold → 5 times
# 2. Average the 5 ROC-AUC scores for each combination
# 3. Pick the combination with the highest average score
# 4. Retrain that winning combination on ALL 733 training samples from scratch
# 5. Store the final model as grid_search.best_estimator_
#
# n_jobs=-1 → uses all available CPU cores to run fits in parallel
grid_search = GridSearchCV(
    estimator  = RandomForestClassifier(random_state=42),
    param_grid = param_grid,
    cv         = cv_strategy,
    scoring    = 'roc_auc',
    n_jobs     = -1,
    verbose    = 1
)

print("Starting hyperparameter search — this may take a few minutes...")
grid_search.fit(X_train, y_train)

print("\nSearch complete.")
print(f"Best parameters : {grid_search.best_params_}")
print(f"Best CV ROC-AUC : {grid_search.best_score_:.4f}")

In [ ]:
# Save and display the full hyperparameter search results
# grid_search.cv_results_ contains the score for every combination tried.
# Saving all 432 rows to CSV allows us to review close competitors and
# understand how sensitive performance is to each hyperparameter.
tuning_df   = pd.DataFrame(grid_search.cv_results_)
tuning_path = os.path.join(EVAL_DIR, f"{MODEL_NAME}_hyperparameter_tuning.csv")
tuning_df.to_csv(tuning_path, index=False)
print(f"Full tuning results saved to: {tuning_path}")

# Display the top 10 winning configurations ranked by validation ROC-AUC
# mean_validation_score → average ROC-AUC across 5 validation folds
#                         Note: scikit-learn internally calls this 'mean_test_score'
#                         — the word 'test' here refers to the validation fold,
#                         NOT the held-out test set locked away in Cell 12
# std_validation_score  → consistency of the score across 5 folds
#                         lower std = more reliable combination
# rank_validation_score → 1 = best combination overall
top10 = (
    tuning_df[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']]
    .sort_values('rank_test_score')
    .head(10)
    .round(4).rename(columns={
        'mean_test_score' : 'mean_validation_score',
        'std_test_score'  : 'std_validation_score',
        'rank_test_score' : 'rank_validation_score'
    })
)
print("\nTop 10 hyperparameter configurations:")
display(top10)

In [ ]:
# Extract the final trained model
# best_estimator_ is NOT one of the 2,160 temporary models from the search.
# After identifying the winning combination, GridSearchCV automatically trained
# a brand new model from scratch using those settings on ALL 733 training samples.
# This final model has seen more data than any fold-level model during the search,
# making it the strongest possible version of the winning configuration.
#
# All subsequent cells use best_rf — it is the only model we evaluate and report.
best_rf = grid_search.best_estimator_

print("Best Random Forest configuration:")
print(best_rf)

---
## Section 3. Evaluation Metrics & Visualizations



### 5-Fold Cross-Validation

In [ ]:
# Stage 1 Evaluation: Stability and Overfitting Check
# We now re-evaluate best_rf using 5-fold cross-validation on the training set.
# This is DIFFERENT from what GridSearchCV did internally:
#   — GridSearchCV used CV to COMPARE 432 combinations (selection bias present)
#   — cross_validate uses CV to MEASURE this one fixed model honestly
#
# By collecting both training scores and validation scores, we can compute
# the Gap: the difference between how well the model fits training data
# vs how well it generalises to data it was not trained on.
#
# A small gap means the model generalises well.
# A large gap means the model memorised training data (overfitting).
cv_results = cross_validate(
    best_rf,
    X_train, y_train,
    cv                 = cv_strategy,
    scoring            = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
    return_train_score = True
)

# Build the comparison table
# Train Mean      → average score when evaluated on the data the model was trained on
# Validation Mean → average score when evaluated on the fold held out for validation
# Gap             → Train Mean - Validation Mean
# Std Dev         → how consistent the validation score is across 5 folds
#                   < 0.03 → stable model
#                   > 0.05 → unstable, results depend too much on which data it sees
cv_summary = pd.DataFrame({
    'Metric'          : ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Train Mean'      : [
        cv_results['train_accuracy'].mean(),
        cv_results['train_precision'].mean(),
        cv_results['train_recall'].mean(),
        cv_results['train_f1'].mean(),
        cv_results['train_roc_auc'].mean()
    ],
    'Validation Mean' : [
        cv_results['test_accuracy'].mean(),
        cv_results['test_precision'].mean(),
        cv_results['test_recall'].mean(),
        cv_results['test_f1'].mean(),
        cv_results['test_roc_auc'].mean()
    ],
    'Gap'             : [
        cv_results['train_accuracy'].mean()  - cv_results['test_accuracy'].mean(),
        cv_results['train_precision'].mean() - cv_results['test_precision'].mean(),
        cv_results['train_recall'].mean()    - cv_results['test_recall'].mean(),
        cv_results['train_f1'].mean()        - cv_results['test_f1'].mean(),
        cv_results['train_roc_auc'].mean()   - cv_results['test_roc_auc'].mean()
    ],
    'Std Dev'         : [
        cv_results['test_accuracy'].std(),
        cv_results['test_precision'].std(),
        cv_results['test_recall'].std(),
        cv_results['test_f1'].std(),
        cv_results['test_roc_auc'].std()
    ]
}).round(4)

print("5-Fold Cross-Validation Results:")
display(cv_summary)

# Save to CSV
cv_path = os.path.join(EVAL_DIR, f"{MODEL_NAME}_cv_results.csv")
cv_summary.to_csv(cv_path, index=False)
print(f"\nSaved to: {cv_path}")

### Final Evaluation on the Held-Out Test Set

In [ ]:
# Stage 2 Evaluation: Final Unbiased Test on Held-Out Data
# This is the first and only time the model encounters the 184 test samples.
# These samples were locked away before any training or tuning began.
# The model had no influence over them — not during training, not during CV,
# not during hyperparameter selection. This makes the result completely unbiased.
#
# predict()       → outputs the final class label (0 or 1) for each patient
# predict_proba() → outputs the probability of being class 1 (Heart Disease)
#                   the [:, 1] takes only the probability for class 1
#                   this is needed to compute ROC-AUC
y_pred      = best_rf.predict(X_test)
y_pred_prob = best_rf.predict_proba(X_test)[:, 1]

# Compute all five evaluation metrics on the 184 test patients
# Accuracy  → out of 184 patients, what percentage were classified correctly?
# Precision → of all patients the model flagged as Heart Disease, how many truly had it?
# Recall    → of all 102 patients who actually had Heart Disease, how many did the model catch?
# F1-Score  → the harmonic mean of Precision and Recall (useful when both matter)
# ROC-AUC   → how well the model separates the two classes across all possible thresholds
#             0.5 = no better than random guessing | 1.0 = perfect separation
test_metrics = pd.DataFrame([{
    'Accuracy' : accuracy_score (y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall'   : recall_score   (y_test, y_pred),
    'F1-Score' : f1_score       (y_test, y_pred),
    'ROC-AUC'  : roc_auc_score  (y_test, y_pred_prob)
}]).round(4)

print("Final Test Set Evaluation Metrics:")
display(test_metrics)

# Save to CSV
metrics_path = os.path.join(EVAL_DIR, f"{MODEL_NAME}_metrics.csv")
test_metrics.to_csv(metrics_path, index=False)
print(f"Saved to: {metrics_path}")

### Confusion Matrix

The confusion matrix below breaks down all 184 test predictions into four
categories. Each number tells us something specific about where the model
succeeded and where it made mistakes. The rows represent what the patient
actually has; the columns represent what the model predicted.

In [ ]:
# Confusion Matrix
# A confusion matrix breaks down all 184 test predictions into 4 categories,
# showing exactly WHERE the model is correct and where it makes mistakes.
#
# Reading the matrix:
#   Rows    = the actual (true) label of the patient
#   Columns = what the model predicted
#
# The 4 cells:
#   TN (top-left)    → model predicted No Disease, patient actually has No Disease ✓
#   FP (top-right)   → model predicted Heart Disease, patient is actually Healthy
#                      consequence: unnecessary further testing, patient anxiety
#   FN (bottom-left) → model predicted No Disease, patient actually has Heart Disease
#                      consequence: missed diagnosis (most dangerous error in medical AI)
#   TP (bottom-right)→ model predicted Heart Disease, patient actually has Heart Disease ✓
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',                              # display counts as integers
    cmap='Blues',
    xticklabels=['No Disease (0)', 'Heart Disease (1)'],
    yticklabels=['No Disease (0)', 'Heart Disease (1)'],
    ax=ax
)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label',      fontsize=12)
ax.set_title('Confusion Matrix — Random Forest', fontsize=13)
plt.tight_layout()

# Save plot
cm_path = os.path.join(PLOTS_DIR, f"{MODEL_NAME}_confusion_matrix.png")
plt.savefig(cm_path, dpi=150)
plt.show()
print(f"Saved to: {cm_path}")

# Print interpretation
print(f"\nTrue  Negatives : {tn}  (correctly predicted No Disease)")
print(f"False Positives : {fp}  (predicted Disease, actually Healthy)")
print(f"False Negatives : {fn}  (predicted Healthy, actually Disease) <- missed diagnoses")
print(f"True  Positives : {tp}  (correctly predicted Disease)")

### ROC Curve

The ROC curve below visualises how well the model separates heart disease
patients from healthy ones across every possible decision threshold (not just the default 0.5). The blue curve represents our model; the grey dashed line represents a model that guesses randomly. The further the blue curve bows toward the top-left corner, the better the model is at correctly identifying sick patients while keeping false alarms low. The AUC (Area Under the Curve) summarises this into a single number (our model scores 0.9210).

In [ ]:
# ROC Curve
# The ROC curve visualises how the model performs at EVERY possible threshold,
# not just the default 0.5. This is especially important in medical systems
# where the threshold may be adjusted by clinicians based on risk tolerance.
#
# X-axis: False Positive Rate (FPR) → proportion of healthy patients falsely flagged
# Y-axis: True Positive Rate (TPR)  → proportion of sick patients correctly caught
#
# Every point on the curve represents a different threshold:
#   — Low threshold (e.g. 0.2) → catch almost all disease cases but many false alarms
#   — High threshold (e.g. 0.8) → very few false alarms but many missed cases
#
# The grey dashed diagonal = random guessing (AUC = 0.50)
# The blue curve = our model — the further it bows toward the top-left, the better
# AUC = area under the curve:
#   0.50 → no better than random  |  0.70–0.80 → acceptable
#   0.80–0.90 → good              |  0.90+ → excellent
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
auc_score   = roc_auc_score(y_test, y_pred_prob)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='steelblue', lw=2,
        label=f'Random Forest (AUC = {auc_score:.4f})')
ax.plot([0, 1], [0, 1], color='grey', linestyle='--', lw=1,
        label='Random Classifier (AUC = 0.50)')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate',  fontsize=12)
ax.set_title('ROC Curve — Random Forest', fontsize=13)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()

# Save plot
roc_path = os.path.join(PLOTS_DIR, f"{MODEL_NAME}_roc_curve.png")
plt.savefig(roc_path, dpi=150)
plt.show()
print(f"Saved to: {roc_path}")

### Feature Importance

The chart below ranks all 16 features by how much they contributed to the
model's predictions. A higher bar means the model relied on that feature more heavily when deciding whether a patient has heart disease or not.

In [ ]:
# Feature Importance
# After training, Random Forest can tell us how much each feature contributed
# to making correct predictions across all trees in the forest.
#
# The importance score is measured by Mean Decrease in Impurity (MDI):
# every time a feature is used to split a node, it reduces the uncertainty
# (impurity) in the resulting groups. Importance = total impurity reduction
# caused by that feature, averaged across all trees.
#
# Higher score → the feature was used more often and more effectively
# Lower score  → the feature contributed little to the predictions
#
# In a medical context this is valuable: it tells us which clinical measurements
# are the strongest predictors of heart disease, which can help clinicians
# prioritise which tests or observations matter most for diagnosis
importances   = best_rf.feature_importances_
feature_names = X.columns

importance_df = (
    pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    .sort_values('Importance', ascending=False)
    .reset_index(drop=True)
)

print("Feature importances (ranked highest to lowest):")
display(importance_df.round(4))

# Bar chart
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data    = importance_df,
    x       = 'Importance',
    y       = 'Feature',
    palette = 'viridis',
    ax      = ax
)
ax.set_title('Feature Importance — Random Forest', fontsize=13)
ax.set_xlabel('Mean Decrease in Impurity (Importance Score)', fontsize=11)
ax.set_ylabel('Feature', fontsize=11)
plt.tight_layout()

# Save plot and CSV
fi_plot_path = os.path.join(PLOTS_DIR, f"{MODEL_NAME}_feature_importance.png")
plt.savefig(fi_plot_path, dpi=150)
plt.show()
print(f"Plot saved to: {fi_plot_path}")

fi_csv_path = os.path.join(EVAL_DIR, f"{MODEL_NAME}_feature_importance.csv")
importance_df.to_csv(fi_csv_path, index=False)
print(f"CSV  saved to: {fi_csv_path}")

### Result Interpretation

#### Best Parameters Found
```
class_weight      = None
max_depth         = 10
max_features      = 'sqrt'
min_samples_leaf  = 4
min_samples_split = 20
n_estimators      = 200
Best CV ROC-AUC   = 0.9256
```

GridSearchCV searched across 864 combinations and selected a forest of 200 trees, each capped at depth 10, requiring at least 20 samples before any split and at least 4 samples at every leaf, considering √16 ≈ 4 features per split, with equal class weights.

**Why these specific values?**

- **n_estimators = 200:** The search found that 200 trees was sufficient for the ensemble to reach a stable consensus on this dataset. Adding more trees (300, 500) did not improve the ROC-AUC score meaningfully, the forest had already captured the available signal with 200 votes.

- **max_depth = 10:** Moderate depth was needed to capture the non-linear relationships confirmed in the EDA, particularly the interactions between ST_Slope, ExerciseAngina, and Oldpeak. Shallower values (5, 8) were too restrictive and missed these patterns. Deeper values caused overfitting by allowing trees to memorise individual patient noise rather than general patterns.

- **min_samples_split = 20:** Each node must contain at least 20 patients before it is allowed to split further. A split backed by 20 patients is statistically far more reliable than one backed by 2 or 5. This prevents the trees from making decisions based on tiny, potentially noisy subgroups.

- **min_samples_leaf = 4:** Every final prediction must be backed by at least 4 patients. This stops trees from creating leaves that represent just one or two unusual patients, which would be memorisation rather than learning.

- **max_features = 'sqrt':** Each split considers only √16 = 4 randomly selected features. This forces diversity between trees where each tree sees a different random subset of features at each node, so no single feature dominates all trees. This diversity is what makes the ensemble stronger than any single decision tree.

- **class_weight = None:** Equal weights produced better results than balanced weights. This confirms that the 55/45 class distribution in this dataset is balanced enough that artificial reweighting introduces unnecessary bias rather than correcting a real imbalance problem.

---

#### Cross-Validation Results

| Metric | Train Mean | Validation Mean | Gap | Std Dev |
|---|---|---|---|---|
| Accuracy | 0.8834 | 0.8513 | 0.0320 | 0.0375 |
| Precision | 0.8747 | 0.8498 | 0.0250 | 0.0544 |
| Recall | 0.9210 | 0.8938 | 0.0272 | 0.0060 |
| F1-Score | 0.8972 | 0.8702 | 0.0270 | 0.0274 |
| ROC-AUC | 0.9637 | 0.9256 | 0.0381 | 0.0185 |

**What the Gap tells us — overfitting assessment:**
All gaps fall between 0.025 and 0.038, well within the acceptable Gap. This means the model performs only slightly better on data it was trained on compared to data it has never seen (a sign of healthy generalisation). The model is not memorising the training data; it is learning patterns that actually transfer to new patients. This result was achieved through iterative hyperparameter tightening across multiple tuning rounds. The gap was originally 0.109 and was systematically reduced by raising min_samples_split, min_samples_leaf, and restricting max_depth until the trees were constrained enough to generalise.

**What the Std Dev tells us — stability assessment:**
Recall has an exceptionally low std dev of 0.006 (the lowest of all metrics). This means the model detects heart disease cases at almost exactly the same rate regardless of which 146 patients make up the validation fold in any given run. This is the most important stability indicator in a medical context, it means the model's ability to catch disease is consistent and not dependent on a lucky split. Precision at 0.054 is slightly higher but still acceptable for a dataset of this size, where a shift of just 8 correct positive predictions can move precision by 5%.

**Why Recall is the most clinically important metric:**
A Validation Recall of 0.8938 means the model catches 89% of actual heart disease cases on average across the 5 training folds. The remaining 11% are false negatives (patients the model incorrectly classifies as healthy). In a medical screening system, these are the most dangerous errors because a missed diagnosis means a sick patient leaves without treatment or follow-up care.

---

#### Test Set Results

| Accuracy | Precision | Recall | F1-Score | ROC-AUC |
|---|---|---|---|---|
| 0.8370 | 0.8529 | 0.8529 | 0.8529 | 0.9210 |

**Accuracy 83.70%:** The model correctly classified 154 out of 184 completely unseen patients. 30 patients were misclassified (15 false positives and 15 false negatives). For a dataset of 917 samples and 16 features, this is a realistic and honest result that reflects genuine learning.

**Precision 85.29%:** When the model predicts Heart Disease, it is correct 85% of the time. Only 15 out of every 100 patients flagged as sick are actually healthy. This means clinicians following up on the model's positive predictions would encounter relatively few unnecessary investigations.

**Recall 85.29%:** The model caught 87 out of 102 actual heart disease patients meaning it missed 15. A recall of 85.3% means roughly 1 in 7 sick patients would not be flagged. While this is a strong result for a model of this size, it reinforces that the model should function as a decision support tool alongside clinical examination rather than a standalone diagnostic system.

**Precision = Recall = F1-Score = 0.8529:** The fact that all three are identical is a notable result. It means the model produced exactly the same number of false positives (15) and false negatives (15) (a perfectly balanced error distribution). This is rare and suggests the model has found a decision boundary that equally weighs the cost of both error types, which is appropriate given that `class_weight=None` was selected, which tells the model to treat both classes equally during training.

**ROC-AUC 0.9210:** This means that if you randomly selected one heart disease patient and one healthy patient from the test set, the model would correctly assign a higher risk score to the sick patient 92.1% of the time (across every possible decision threshold). An AUC above 0.90 is considered excellent in medical AI literature and confirms the model has genuinely learned to separate the two classes rather than memorising patterns from the training data.

**CV vs Test agreement — representativeness check:**

| Metric | CV Validation | Test Set | Difference |
|---|---|---|---|
| Accuracy | 0.8513 | 0.8370 | 0.014 |
| Precision | 0.8498 | 0.8529 | +0.003 |
| Recall | 0.8938 | 0.8529 | 0.041 |
| F1-Score | 0.8702 | 0.8529 | 0.017 |
| ROC-AUC | 0.9256 | 0.9210 | 0.005 |

All differences are small (the largest is recall at 0.041). This close agreement confirms two things: first, the 184 test samples are representative of the full dataset meaning that the model did not benefit from an unusually easy test set. Second, there is no data leakage. If the test set had been indirectly influenced by training or tuning, test scores would be artificially inflated. The fact that most test scores are slightly lower than CV scores is exactly what a healthy, unbiased evaluation looks like.

---

#### Confusion Matrix
```
                      Predicted No Disease    Predicted Heart Disease
Actual No Disease          TN = 67                  FP = 15
Actual Heart Disease       FN = 15                  TP = 87
```

**True Negatives (67):** The model correctly identified 67 out of 82 healthy patients (81.7% of healthy patients were correctly cleared with no further action needed).

**False Positives (15):** 15 healthy patients were incorrectly flagged as having heart disease (18.3% false alarm rate among healthy patients). These patients would undergo unnecessary further testing, causing anxiety and additional healthcare cost. However, in a screening context this is the more acceptable error type compared to a missed diagnosis.

**False Negatives (15):** 15 patients who actually had heart disease were classified as healthy (14.7% missed diagnosis rate among sick patients). These are the most dangerous errors, each represents a patient who could leave without receiving necessary treatment or follow-up. Notably, FP and FN are equal at 15 each, meaning the model's errors are perfectly balanced between the two error types. If a clinical team wanted to reduce missed diagnoses at the cost of more false alarms, they could lower the decision threshold below 0.5 without retraining the model.

**True Positives (87):** The model correctly identified 87 out of 102 heart disease patients (85.3% detection rate, consistent with the recall metric reported above).

---

#### Feature Importance

| Rank | Feature | Importance |
|---|---|---|
| 1 | ST_Slope_Up | 0.2645 |
| 2 | ST_Slope_Flat | 0.1813 |
| 3 | ExerciseAngina | 0.1303 |
| 4 | Oldpeak | 0.0983 |
| 5 | MaxHR | 0.0780 |
| 6 | ChestPainType_ATA | 0.0497 |
| 7 | Sex | 0.0438 |
| ... | ... | ... |
| 12–16 | RestingECG, Chol_category | < 0.008 each |


The top 5 features alone account for **75.24%** of the model's total predictive power, and the
top 7 features collectively reach **84.59%**, meaning less than half the features drive nearly
85% of every prediction. Notably, the five highest-ranked features (ST_Slope, ExerciseAngina,
Oldpeak, and MaxHR) are all measurements taken during an exercise stress test. The fact that
the model, trained purely on patient data with no medical knowledge built in, arrived at the
same hierarchy of importance that clinical guidelines have established over decades is strong
evidence that it learned real, meaningful patterns rather than statistical coincidences.

### ST_Slope_Up & ST_Slope_Flat — Ranks 1 & 2 (Combined 44.58%)
As established in the EDA, a flat or downsloping ST segment during exercise indicates the heart
is not receiving enough blood, while an upsloping segment reflects a normal cardiac response.
Lauer et al. (1996) confirmed that ST segment direction during exercise is one of the most
reliable non-invasive indicators of heart disease and the model independently learned the same.
The Up and Flat categories cover **854 out of 917 patients (93%)**: **80.3% of Up patients are
healthy** and **82.8% of Flat patients have heart disease** this explains why these two columns account for
**44.6% of total feature importance**.
The Down category (77.8% disease rate) was dropped as the reference category during one-hot
encoding. With only **63 patients (6.9%)**, its signal is absorbed indirectly by the other
two columns.
> Lauer, M.S., et al. (1996). *Circulation*, 93(8), 1520–1526.
> https://doi.org/10.1161/01.cir.93.8.1520

### ExerciseAngina — Rank 3 (13.03%)
As established in the EDA, exercise-induced angina is a direct warning sign that the heart is
not receiving enough blood during physical demand. Weiner et al. (1978) found coronary artery
disease in 95% of patients who experienced chest pain alongside abnormal ECG changes during
exercise testing, and the model independently identified the same pattern.
Of the 917 patients, **371 (40.5%) reported chest pain** and **546 (59.5%) did not**. Disease
rates: **85.2% for Yes versus 35.1% for No**. This is a gap of over 50 percentage points , contributing
**13.0% of total feature importance**.
> Weiner, D.A., et al. (1978). *American Heart Journal*, 96(4), 458–462.
> https://doi.org/10.1016/0002-8703(78)90155-2

### Oldpeak — Rank 4 (9.83%)
As established in the EDA, Oldpeak measures ST-segment depression during exercise (the higher
the value, the more the heart struggled under physical stress). Miranda et al. (1991) confirmed that
greater ST depression during exercise is a reliable marker for coronary artery disease,
the bigger the dip, the more confident the model becomes that the patient has heart disease.
As noted in the EDA pair plot, heart disease cases tend to concentrate at Oldpeak values greater
than 1.5. Among the **100 patients with Oldpeak above 2.0**, **91 (91%) had heart disease**.
This explains why Oldpeak is contributing **9.8% of total feature importance**.
> Miranda, C.P., et al. (1991). *American Heart Journal*, 122(6), 1617–1628.
> https://doi.org/10.1016/0002-8703(91)90279-q


### MaxHR — Rank 5 (7.80%)
As established in the EDA, a diseased heart cannot reach normal peak heart rates during exercise
due to narrowed arteries limiting blood supply. Sandvik et al. (1995) followed nearly
2,000 men for 16 years and confirmed that a lower maximum exercise heart rate is a strong,
independent predictor of cardiovascular death which is consistent with what the model found.
As noted in the EDA pair plot, heart disease cases increase as MaxHR decreases, while healthy
cases increase as MaxHR increases contributing **7.8% of total feature importance**.
> Sandvik, L., et al. (1995). *Coronary Artery Disease*, 6(8), 667–679.
> https://doi.org/10.1097/00019501-199508000-00012

### ChestPainType_ATA — Rank 6 (4.97%)
As established in the EDA, ASY (Asymptomatic) patients feel no chest pain despite possible
underlying disease. ATA (Atypical Angina) does not meet full ischemic criteria and is less
associated with coronary artery disease.

| Chest Pain Type | Disease Rate |
|----------------|-------------|
| ASY (Asymptomatic) | 79.0% |
| TA (Typical Angina) | 43.5% |
| NAP (Non-Anginal Pain) | 35.1% |
| ATA (Atypical Angina) | **13.9%** |

ATA is the clearest healthy signal (**86.1% healthy**). ASY patients appear as all three chest
pain columns set to zero (the dropped reference) so the model reads all zeros as "no chest
pain," strongly associated with disease, contributing **5.0% of total feature importance**.

### Sex — Rank 7 (4.38%)
As established in the EDA, men develop coronary artery disease earlier than women.
The dataset contains **724 males (78.9%)** and **193 females (21.1%)**.
Disease rates: **63.1% for males versus 25.9% for females**.

However, it is important to interpret this carefully. The disease rate of 63.1% for males
is calculated within the male group alone (it is a fair comparison regardless of how many
males are in the dataset). But the reason Sex ranks as high as it does in feature importance
is partly a reflection of the dataset composition. Because the model encounters male
patients in 78.9% of all cases, the male-associated disease signal gets applied to the large
majority of predictions. In a dataset with a more balanced sex distribution, Sex might rank lower.
The model learned from the data it was given, and in this data, being male is both strongly and
frequently associated with heart disease, earning Sex 4.4% of the total feature importance.

### Why Cholesterol Ranked Low
As established in the EDA, high LDL cholesterol is a known risk factor for coronary artery
disease. Interestingly, the signal does exist in the data: among the **746 patients with valid
non-zero readings**, healthy patients averaged **238.77 mg/dL** versus **251.06 mg/dL**.
However, **171 patients (18.6%)** had a value of exactly 0, indicating missing data that was
imputed with the median during preprocessing. From the model's perspective, 171 patients who
are otherwise completely different from each other (different ages, different symptoms, different
outcomes) now look identical on this one feature. The relationship exists however the data quality
does not allow the model to fully see it.


### Overall clinical validity
The four most important features (ST slope, exercise angina, ST depression (Oldpeak), and maximum heart rate) are all measurements taken from or during an exercise stress test. This is one of the most widely used non-invasive tools in cardiology for diagnosing heart disease, and these are exactly the measurements cardiologists examine most closely when interpreting the results. The fact that the model, trained purely on patient data with no medical knowledge built in, arrived at the same hierarchy of importance that clinical guidelines have established over decades is strong evidence that it learned real, meaningful patterns rather than statistical coincidences.

# XGBoost

## Section 1. Rationale for Choosing XGBoost:

XGBoost (Extreme Gradient Boosting) is an advanced gradient boosting algorithm
that is widely used in machine learning for both classification and regression
tasks. It is built upon decision trees, which makes it well-suited for capturing
complex patterns in data, especially non-linear relationships. XGBoost works by
constructing multiple decision trees in a sequential manner, where each new tree
aims to correct the mistakes of the previous trees, making it a strong ensemble
model (Chen & Guestrin, 2016).

### **Why XGBoost is a Good Fit for Our Data?**

#### i. Dataset Characteristics (Size, Feature Types, and Linearity)

Many features (such as Oldpeak and MaxHR) exhibit non-linear relationships with
the target variable HeartDisease. For example, FastingBS shows only a moderate
correlation (0.27), suggesting that the relationship is not linear. As a
tree-based model, XGBoost captures non-linear patterns by recursively
partitioning the data, allowing it to model complex relationships that linear
models cannot (Chen & Guestrin, 2016).

Additionally, features such as Oldpeak contain extreme values. Linear models
are sensitive to outliers, which may bias predictions. In contrast, XGBoost is
more robust because decision trees split data using thresholds rather than
distances, reducing the influence of extreme values (Hastie et al., 2009).

Since the final dataset is relatively small, deep learning models would be prone
to overfitting. XGBoost performs well on small-to-medium structured tabular
datasets due to its boosting mechanism and regularisation (Chen & Guestrin,
2016). Furthermore, XGBoost effectively handles structured tabular data
containing continuous, binary, and one-hot encoded categorical features.

#### ii. Problem Type (Binary Classification)

The task is to predict the presence of heart disease, which is a binary
classification problem.

XGBoost is highly suitable for binary classification tasks because:

- It supports binary classification natively via the `binary:logistic`
  objective function (Chen & Guestrin, 2016).
- It optimises classification performance using gradient boosting.
- It performs well on structured/tabular medical datasets (Budholiya et al., 2022).

Thus, the nature of the problem aligns well with XGBoost's capabilities.

#### iii. Model Strengths and Weaknesses for Our Specific Problem

**Strengths:**

- Handles non-linear relationships effectively through tree-based decision
  boundaries (Chen & Guestrin, 2016).
- Robust to outliers, which were initially observed in features such as Oldpeak
  and Cholesterol during exploratory analysis (Hastie et al., 2009).
- Strong performance on structured tabular data, such as clinical datasets
  (Budholiya et al., 2022).
- Feature interaction modelling, which is important in medical prediction
  problems where variables may interact in complex ways (Chen & Guestrin, 2016).

**Weaknesses:**

- Can be computationally more expensive than simpler models.
- May require hyperparameter tuning for optimal performance
  (Bergstra & Bengio, 2012).

However, considering our dataset characteristics and binary classification task,
the strengths of XGBoost outweigh its weaknesses, making it a strong and
suitable choice for predicting heart disease.

---

> **References:**
>
> Chen, T., & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System.
> *Proceedings of the 22nd ACM SIGKDD*, 785–794.
> https://doi.org/10.1145/2939672.2939785
>
> Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of
> Statistical Learning* (2nd ed.). Springer.
> https://hastie.su.domains/ElemStatLearn/
>
> Bergstra, J., & Bengio, Y. (2012). Random Search for Hyper-Parameter
> Optimization. *Journal of Machine Learning Research*, 13, 281–305.
> http://www.jmlr.org/papers/v13/bergstra12a.html
>
> Budholiya, K., Shrivastava, S.K., & Sharma, V. (2022). An optimized
> XGBoost based diagnostic system for effective prediction of heart disease.
> *Journal of King Saud University — Computer and Information Sciences*,
> 34(7), 4514–4523.
> https://doi.org/10.1016/j.jksuci.2020.10.013

## Section 2. Implementation & Training

### Imports & Setup

In [ ]:
import subprocess
subprocess.run(["pip", "install", "xgboost"], check=True)
subprocess.run(["pip", "install", "seaborn"], check=True)

# Standard libraries
import pickle
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
warnings.filterwarnings('ignore')

# Model
import xgboost as xgb

# Splitting & cross-validation
from sklearn.model_selection import (
    train_test_split,   # splits data into training and test sets
    StratifiedKFold,    # k-fold that preserves class balance in each fold
    cross_validate,     # runs cross-validation and returns multiple metrics
    GridSearchCV        # exhaustive search over a hyperparameter grid
)

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,     # proportion of correct predictions
    precision_score,    # of all predicted positives, how many are truly positive
    recall_score,       # of all actual positives, how many were correctly predicted
    f1_score,           # harmonic mean of precision and recall
    confusion_matrix,   # table showing TP, TN, FP, FN counts
    roc_auc_score,      # area under the ROC curve
    roc_curve           # false positive rate vs true positive rate at all thresholds
)

# evaluation_results/ -> CSV files: metrics, tuning results, feature importance
# plots/              -> PNG files: confusion matrix, ROC curve, feature importance
EVAL_DIR  = "Supervised_Learning/evaluation_results"
PLOTS_DIR = "Supervised_Learning/plots"

os.makedirs(EVAL_DIR,  exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# Model name prefix
# Used in every saved file name so outputs are clearly labelled per model
MODEL_NAME = "xgboost"

print("Libraries imported successfully.")
print(f"Evaluation results → {EVAL_DIR}")
print(f"Plots              → {PLOTS_DIR}")

### Data Loading

In [ ]:
# Load the preprocessed dataset
# This CSV was produced by the EDA notebook — it contains cleaned, encoded,
# and normalised features ready for modeling. No further preprocessing is needed.
DATA_PATH = "Dataset/preprocessed_heart_data.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

### Feature & Target Split

In [ ]:
# Separate input features from the target variable
# X -> all feature columns (what the model uses to make predictions)
# y -> the target column   (what the model is trying to predict)
TARGET = "HeartDisease"   # 0 = No Disease, 1 = Heart Disease

X = df.drop(columns=[TARGET])
y = df[TARGET]

print(f"Features (X): {X.shape[1]} columns")
print(f"\nTarget (y) distribution:")
print(y.value_counts().rename({0: 'No Disease (0)', 1: 'Heart Disease (1)'}))

### Train/Test Split

In [ ]:
# Create or load the 80/20 train/test split
# This cell implements the first and most critical step of our methodology:
# locking away 20% of the data before any training or tuning begins.
#
# IMPORTANT: This split is created ONCE and shared across all models.
# If each model used a different split, performance differences could be due
# to the split rather than the model — making comparison unfair.
# Saving and loading the same indices guarantees a completely fair comparison.
SPLIT_PATH = "split_indices.pkl"

if not os.path.exists(SPLIT_PATH):
    # stratify=y   → preserves the class ratio in both sets
    # random_state → fixes the random seed so the split is reproducible
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    split_indices = {
        "train_indices": X_train.index.tolist(),
        "test_indices" : X_test.index.tolist()
    }
    with open(SPLIT_PATH, "wb") as f:
        pickle.dump(split_indices, f)

    print("Split created and saved to split_indices.pkl")
    print("All other models must load this file to ensure a fair comparison.")
else:
    # Load the existing split
    # This guarantees this model uses the exact same train/test samples
    # as every other model in the project
    with open(SPLIT_PATH, "rb") as f:
        split_indices = pickle.load(f)

    X_train = X.iloc[split_indices["train_indices"]]
    X_test  = X.iloc[split_indices["test_indices"]]
    y_train = y.iloc[split_indices["train_indices"]]
    y_test  = y.iloc[split_indices["test_indices"]]

    print("Existing split loaded from split_indices.pkl")
    print("This XGBoost model uses the EXACT same train/test samples as Random Forest.")

print(f"\nTraining set : {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test set     : {X_test.shape[0]}  samples ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\nClass distribution in training set:")
print(y_train.value_counts().rename({0: 'No Disease', 1: 'Heart Disease'}))
print(f"\nClass distribution in test set:")
print(y_test.value_counts().rename({0: 'No Disease', 1: 'Heart Disease'}))

### XGBoost Hyperparameter Selection

XGBoost operates through a sequential boosting process, where each tree is built
to correct the mistakes of the one before it. This makes its hyperparameters
particularly interconnected, adjusting one setting can amplify or dampen the
effect of another. The challenge is not tuning each parameter in isolation, but
finding a combination that balances learning speed, tree complexity, and
generalization across the specific dataset at hand.

### Why These Specific Hyperparameters Were Selected for Tuning

XGBClassifier exposes over 20 configurable parameters. From these, 7 were selected
for tuning based on their direct influence on the two core failure modes of boosted
models: **overfitting** and **unstable learning**. The rest were left at their
defaults, either because they overlap with parameters already being tuned, because
meaningful adjustment requires domain-specific knowledge beyond the scope of this
work, or because they govern computational behavior rather than predictive performance.

**Tuned parameters:**

| Parameter | Why Tuned |
|-----------|-----------|
| `n_estimators` | Number of boosting rounds to run — More rounds give the model more opportunities to correct errors; the default is often insufficient for medical datasets where patterns are subtle |
| `max_depth` | Maximum depth allowed per tree — Deep trees tend to memorise the training data rather than learn from it; depth was constrained to encourage generalisation |
| `learning_rate` | Step size applied at each boosting round — A high default value can push the model toward premature convergence; reducing it allows more gradual and robust learning |
| `subsample` | Proportion of training samples drawn per round — Training on the full dataset every round increases the risk of overfitting; random sampling introduces beneficial variance |
| `colsample_bytree` | Proportion of features sampled per tree — Using all features at every split encourages reliance on dominant variables; sampling promotes diversity across trees |
| `min_child_weight` | Minimum total sample weight required to form a leaf — A low default permits splits based on very few observations; raising this threshold ensures splits reflect genuine patterns |
| `scale_pos_weight` | Relative weight assigned to the positive class — The dataset has a 55/45 class split; this parameter was included to allow GridSearchCV to determine whether correction was warranted |

**Parameters not tuned:**

The remaining parameters were excluded for one of four reasons: they address the
same concerns as tuned parameters and would introduce redundancy (`gamma`,
`reg_alpha`, `reg_lambda`, `max_delta_step`); they define core algorithmic
behavior that should remain unchanged (`objective`, `booster`, `tree_method`);
they serve evaluation or early stopping purposes rather than shaping the model
itself (`eval_metric`, `early_stopping_rounds`); or they control runtime
configuration without influencing predictions (`n_jobs`, `random_state`,
`verbosity`).



> **Reference:** > XGBoost Developers. *XGBoost Parameters*.
> https://xgboost.readthedocs.io/en/stable/parameter.html

In [ ]:
# Step 1: Define the hyperparameter search grid
# param_grid is a menu of options for GridSearchCV to try.
# GridSearchCV will test every possible combination of these values —
# 4×4×4×3×3×3×2 = 3,456 combinations × 5 folds = 17,280 model fits in total.
# Each combination is scored using 5-fold cross-validation on the training set.
# The combination with the highest average recall score wins.
param_grid = {
    # Number of boosting rounds (trees) — similar to n_estimators in RF
    # More trees = more opportunities to correct previous errors
    # Lower learning rate requires more trees to converge
    'n_estimators': [100, 200, 300, 500],

    # Maximum tree depth — controls model complexity
    # Shallower trees (3-6) reduce overfitting, deeper trees (9-12) capture more patterns
    # For recall, slightly deeper trees help catch more positive cases
    'max_depth': [3, 6, 9, 12],

    # Learning rate (shrinkage) — how much each tree contributes to the final prediction
    # Lower values (0.01-0.05) = slower learning but better generalisation
    # Must be balanced with n_estimators: low rate needs more trees
    'learning_rate': [0.01, 0.05, 0.1, 0.2],

    # Subsample ratio of training instances per tree (like bootstrap in RF)
    # e.g. 0.8 = randomly use 80% of training data for each tree
    # Introduces randomness to prevent overfitting
    'subsample': [0.7, 0.8, 1.0],

    # Subsample ratio of features used per tree (like max_features in RF)
    # e.g. 0.8 = randomly use 80% of features for each tree
    # Reduces correlation between trees and prevents overfitting
    'colsample_bytree': [0.7, 0.8, 1.0],

    # Minimum sum of instance weights needed in a leaf node
    # Higher values = more conservative model, fewer splits → reduces overfitting
    # Similar to min_samples_leaf in Random Forest
    'min_child_weight': [1, 3, 5],

    # Balances positive/negative class weights during training
    # 1 = no adjustment (balanced dataset)
    # ratio = neg/pos count → upweights the minority class to improve recall
    'scale_pos_weight': [1, (len(y_train[y_train==0]) / len(y_train[y_train==1]))]
}

total_combinations = np.prod([len(v) for v in param_grid.values()])
print(f"Total combinations : {total_combinations}")
print(f"Total fits         : {total_combinations} × 5 folds = {total_combinations*5}")

### GridSearchCV & Model Training

In [ ]:
# Step 2: Define the cross-validation strategy
# This tells GridSearchCV HOW to evaluate each combination fairly:
# — Split the training samples into 5 equal folds
# — Each fold takes a turn being the validation set exactly once
# — Stratified: every fold preserves the class ratio of the full training set
# — shuffle + random_state=42: same fold assignments every run → reproducible results
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Step 3: Run the exhaustive search
# GridSearchCV ties all three components together:
#   estimator  → the model to train (XGBClassifier)
#   param_grid → the combinations to try (3,456 total)
#   cv         → how to evaluate each combination (5-fold stratified CV)
#   scoring    → what metric to optimise (recall)
#
# Why recall and not ROC-AUC or accuracy?
# In a medical context, missing a true disease case (False Negative) is far
# more dangerous than a false alarm (False Positive). Recall directly measures
# how many actual disease cases the model catches — optimising for it minimises
# the most dangerous type of error in heart disease prediction.
#
# What happens when .fit() is called:
# 1. For each of 3,456 combinations → train on 4 folds, validate on 1 fold → 5 times
# 2. Average the 5 recall scores for each combination
# 3. Pick the combination with the highest average recall
# 4. Retrain that winning combination on ALL training samples from scratch
# 5. Store the final model as grid_search.best_estimator_
#
# n_jobs=-1 → uses all available CPU cores to run fits in parallel
grid_search = GridSearchCV(
    estimator=xgb.XGBClassifier(
        random_state=42,
        objective='binary:logistic',  # Binary classification task
        eval_metric='aucpr',          # Area under Precision-Recall curve
                                      # Better than AUC for recall-focused tasks
        verbosity=0                   # Suppress XGBoost internal logs
    ),
    param_grid=param_grid,            # Full grid → all 3,456 combinations tested
    cv=cv_strategy,                   # 5-fold stratified CV per combination
    scoring='recall',                 # Optimise directly for recall
    n_jobs=-1,                        # Use all CPU cores in parallel
    verbose=1                         # Print progress during search
)

print(f"Starting XGBoost GridSearchCV ({total_combinations} combinations × 5 folds = {total_combinations*5} fits)...")
print("Optimising for: RECALL — minimising missed Heart Disease cases\n")

grid_search.fit(X_train, y_train)

print("\nSearch complete.")
print(f"Best parameters : {grid_search.best_params_}")
print(f"Best CV Recall  : {grid_search.best_score_:.4f}")

### Tuning Results & Best Model Extraction

In [ ]:
# Save and display the full hyperparameter search results
# grid_search.cv_results_ contains the score for every combination tried.
# Saving all 3,456 rows to CSV allows us to review close competitors and
# understand how sensitive performance is to each hyperparameter.
tuning_df   = pd.DataFrame(grid_search.cv_results_)
tuning_path = os.path.join(EVAL_DIR, f"{MODEL_NAME}_hyperparameter_tuning.csv")
tuning_df.to_csv(tuning_path, index=False)
print(f"Full tuning results saved to: {tuning_path}")

# Display the top 10 winning configurations ranked by validation recall
# mean_validation_score → average recall across 5 validation folds
#                         Note: scikit-learn internally calls this 'mean_test_score'
#                         — the word 'test' here refers to the validation fold,
#                         NOT the held-out test set locked away
# std_validation_score  → consistency of the score across 5 folds
#                         lower std = more reliable combination
# rank_validation_score → 1 = best combination overall
top10 = (
    tuning_df[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']]
    .sort_values('rank_test_score')
    .head(10)
    .round(4)
    .rename(columns={
        'mean_test_score' : 'mean_validation_score',
        'std_test_score'  : 'std_validation_score',
        'rank_test_score' : 'rank_validation_score'
    })
)
print("\nTop 10 hyperparameter configurations:")
display(top10)

# Extract the final trained model
# best_estimator_ is NOT one of the 17,280 temporary models from the search.
# After identifying the winning combination, GridSearchCV automatically trained
# a brand new model from scratch using those settings on ALL training samples.
# This final model has seen more data than any fold-level model during the search,
# making it the strongest possible version of the winning configuration.
#
# All subsequent cells use best_xgb — it is the only model we evaluate and report.
best_xgb = grid_search.best_estimator_

print("\nBest XGBoost configuration:")
print(best_xgb)


## Section 3. Evaluation Metrics & Visualizations



### 5-Fold Cross-Validation

In [ ]:
# Stage 1 Evaluation: Stability and Overfitting Check
# We now re-evaluate best_xgb using 5-fold cross-validation on the training set.
# This is DIFFERENT from what GridSearchCV did internally:
#   — GridSearchCV used CV to COMPARE 3,456 combinations (selection bias present)
#   — cross_validate uses CV to MEASURE this one fixed model honestly
#
# By collecting both training scores and validation scores, we can compute
# the Gap: the difference between how well the model fits training data
# vs how well it generalises to data it was not trained on.
#
# A small gap means the model generalises well.
# A large gap means the model memorised training data (overfitting).
cv_results = cross_validate(
    best_xgb,
    X_train, y_train,
    cv                 = cv_strategy,
    scoring            = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
    return_train_score = True
)

# Build the comparison table
# Train Mean      → average score when evaluated on the data the model was trained on
# Validation Mean → average score when evaluated on the fold held out for validation
# Gap             → Train Mean - Validation Mean
# Std Dev         → how consistent the validation score is across 5 folds
#                   < 0.03 → stable model
#                   > 0.05 → unstable
cv_summary = pd.DataFrame({
    'Metric'          : ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Train Mean'      : [
        cv_results['train_accuracy'].mean(),
        cv_results['train_precision'].mean(),
        cv_results['train_recall'].mean(),
        cv_results['train_f1'].mean(),
        cv_results['train_roc_auc'].mean()
    ],
    'Validation Mean' : [
        cv_results['test_accuracy'].mean(),
        cv_results['test_precision'].mean(),
        cv_results['test_recall'].mean(),    # ← Most important for our goal
        cv_results['test_f1'].mean(),
        cv_results['test_roc_auc'].mean()
    ],
    'Gap'             : [
        cv_results['train_accuracy'].mean()  - cv_results['test_accuracy'].mean(),
        cv_results['train_precision'].mean() - cv_results['test_precision'].mean(),
        cv_results['train_recall'].mean()    - cv_results['test_recall'].mean(),
        cv_results['train_f1'].mean()        - cv_results['test_f1'].mean(),
        cv_results['train_roc_auc'].mean()   - cv_results['test_roc_auc'].mean()
    ],
    'Std Dev'         : [
        cv_results['test_accuracy'].std(),
        cv_results['test_precision'].std(),
        cv_results['test_recall'].std(),
        cv_results['test_f1'].std(),
        cv_results['test_roc_auc'].std()
    ]
}).round(4)

print("5-Fold Cross-Validation Results (Training vs Validation):")
display(cv_summary)

# Save to CSV
cv_path = os.path.join(EVAL_DIR, f"{MODEL_NAME}_cv_results.csv")
cv_summary.to_csv(cv_path, index=False)
print(f"\nSaved to: {cv_path}")

### Final Evaluation on the Held-Out Test Set

In [ ]:
# Stage 2 Evaluation: Final Unbiased Test on Held-Out Data
# This is the first and only time the model encounters the test samples.
# These samples were locked away before any training or tuning began.
# The model had no influence over them — not during training, not during CV,
# not during hyperparameter selection. This makes the result completely unbiased.
#
# predict()       → outputs the final class label (0 or 1) for each patient
# predict_proba() → outputs the probability of being class 1 (Heart Disease)
#                   the [:, 1] takes only the probability for class 1
#                   this is needed for threshold tuning and ROC-AUC
print("\n" + "-"*40)
print("TEST SET EVALUATION")
print("-"*40)

y_pred      = best_xgb.predict(X_test)
y_pred_prob = best_xgb.predict_proba(X_test)[:, 1]

# Threshold Optimisation
# Default threshold (0.5) is not always optimal for recall.
# precision_recall_curve returns precision/recall values at every possible threshold.
# We search for the lowest threshold where:
#   - Recall    >= 0.88 → catch at least 88% of disease cases
#   - Precision >= 0.80 → keep false alarms under control
# This is justified because in medical diagnosis, missing a true disease case
# (False Negative) is far more dangerous than a false alarm (False Positive)
print("\n" + "-"*40)
print("OPTIMISING THRESHOLD FOR BETTER RECALL")
print("-"*40)

from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_prob)

# precision_recall_curve returns one more value than thresholds → align lengths
thresholds = thresholds[:len(recalls) - 1]
precisions = precisions[:len(recalls) - 1]
recalls    = recalls   [:len(recalls) - 1]

optimal_threshold  = 0.5
best_recall_val    = recall_score   (y_test, y_pred)
best_precision_val = precision_score(y_test, y_pred)

print(f"Default threshold (0.50) → Recall={best_recall_val:.4f}, Precision={best_precision_val:.4f}")

for i, threshold in enumerate(thresholds):
    if recalls[i] >= 0.88 and precisions[i] >= 0.80:
        optimal_threshold  = threshold
        best_recall_val    = recalls[i]
        best_precision_val = precisions[i]
        break

print(f"Optimal threshold ({optimal_threshold:.3f}) → Recall={best_recall_val:.4f}, Precision={best_precision_val:.4f}")

# Apply the optimal threshold to generate final predictions
y_pred_final = (y_pred_prob >= optimal_threshold).astype(int)

# Compute all five evaluation metrics on the test patients
# Accuracy  → out of all patients, what percentage were classified correctly?
# Precision → of all patients the model flagged as Heart Disease, how many truly had it?
# Recall    → of all patients who actually had Heart Disease, how many did the model catch?
# F1-Score  → the harmonic mean of Precision and Recall
# ROC-AUC   → how well the model separates the two classes across all thresholds
#             0.5 = no better than random guessing | 1.0 = perfect separation
print("\n" + "="*40)
print("FINAL RESULTS WITH OPTIMAL THRESHOLD")
print("="*40)

test_metrics = pd.DataFrame([{
    'Accuracy' : accuracy_score (y_test, y_pred_final),
    'Precision': precision_score(y_test, y_pred_final),
    'Recall'   : recall_score   (y_test, y_pred_final),   # ← Primary goal
    'F1-Score' : f1_score       (y_test, y_pred_final),
    'ROC-AUC'  : roc_auc_score  (y_test, y_pred_prob)     # Always uses probabilities
}]).round(4)

print("Final Test Set Evaluation Metrics:")
display(test_metrics)

# Save to CSV
metrics_path = os.path.join(EVAL_DIR, f"{MODEL_NAME}_metrics.csv")
test_metrics.to_csv(metrics_path, index=False)
print(f"Saved to: {metrics_path}")

### Confusion Matrix

The confusion matrix below breaks down all 184 test predictions into four categories.
Each number tells us something specific about where the model succeeded and where it
made mistakes. The rows represent what the patient actually has; the columns represent
what the model predicted.

Unlike the default threshold of 0.5, an optimal threshold of 0.199 was applied to
maximise recall — ensuring the model catches as many true Heart Disease cases as
possible, since a missed diagnosis is far more dangerous than a false alarm in a
medical context.

In [ ]:
# Confusion Matrix
# A confusion matrix breaks down all test predictions into 4 categories,
# showing exactly WHERE the model is correct and where it makes mistakes.
#
# Reading the matrix:
#   Rows    = the actual (true) label of the patient
#   Columns = what the model predicted
#
# The 4 cells:
#   TN (top-left)     → model predicted No Disease, patient actually has No Disease ✓
#   FP (top-right)    → model predicted Heart Disease, patient is actually Healthy
#                       consequence: unnecessary further testing, patient anxiety
#   FN (bottom-left)  → model predicted No Disease, patient actually has Heart Disease
#                       consequence: missed diagnosis (most dangerous error in medical AI)
#   TP (bottom-right) → model predicted Heart Disease, patient actually has Heart Disease ✓
cm = confusion_matrix(y_test, y_pred_final)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Oranges',                                        # Distinct from RF's Blues
    xticklabels=['No Disease (0)', 'Heart Disease (1)'],
    yticklabels=['No Disease (0)', 'Heart Disease (1)'],
    ax=ax
)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label',      fontsize=12)
ax.set_title(f'Confusion Matrix — XGBoost (Threshold={optimal_threshold:.3f})', fontsize=13)
plt.tight_layout()

# Save plot
cm_path = os.path.join(PLOTS_DIR, f"{MODEL_NAME}_confusion_matrix.png")
plt.savefig(cm_path, dpi=150)
plt.show()
print(f"Saved to: {cm_path}")

print(f"\nTrue  Negatives (TN): {tn}  — correctly predicted No Disease")
print(f"False Positives (FP): {fp}  — predicted Disease, actually Healthy (false alarm)")
print(f"False Negatives (FN): {fn}  — MISSED Disease cases ← most critical to minimize")
print(f"True  Positives (TP): {tp}  — correctly predicted Disease")


### ROC Curve

The ROC curve below visualises how well the model separates heart disease
patients from healthy ones across every possible decision threshold (not just
the default 0.5). The orange curve represents our XGBoost model; the grey
dashed line represents a model that guesses randomly. The further the orange
curve bows toward the top-left corner, the better the model is at correctly
identifying sick patients while keeping false alarms low. The AUC (Area Under
the Curve) summarises this into a single number, our model scores **0.9285**,
which indicates excellent discrimination between the two classes.

In [ ]:
# ROC Curve
# The ROC curve visualises how the model performs at EVERY possible threshold,
# not just the default 0.5. This is especially important in medical systems
# where the threshold may be adjusted by clinicians based on risk tolerance.
#
# X-axis: False Positive Rate (FPR) → proportion of healthy patients falsely flagged
# Y-axis: True Positive Rate (TPR)  → proportion of sick patients correctly caught
#
# Every point on the curve represents a different threshold:
#   — Low threshold (e.g. 0.2)  → catch almost all disease cases but many false alarms
#   — High threshold (e.g. 0.8) → very few false alarms but many missed cases
#
# The grey dashed diagonal = random guessing (AUC = 0.50)
# The orange curve = our model — the further it bows toward the top-left, the better
# AUC = area under the curve:
#   0.50 → no better than random  |  0.70–0.80 → acceptable
#   0.80–0.90 → good              |  0.90+ → excellent
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
auc_score   = roc_auc_score(y_test, y_pred_prob)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='darkorange', lw=2,
        label=f'XGBoost (AUC = {auc_score:.4f})')
ax.plot([0, 1], [0, 1], color='grey', linestyle='--', lw=1,
        label='Random Classifier (AUC = 0.50)')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curve — XGBoost', fontsize=13)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()

# Save plot
roc_path = os.path.join(PLOTS_DIR, f"{MODEL_NAME}_roc_curve.png")
plt.savefig(roc_path, dpi=150)
plt.show()
print(f"Saved to: {roc_path}")

#### Feature Importance

The chart below ranks all 16 features by how much they contributed to the
model's predictions. A higher bar means the model relied on that feature more
heavily when deciding whether a patient has heart disease or not. Importance
is measured by **Gain**, the average improvement in prediction accuracy that
a feature brings each time it is used to split a node across all trees.

In [ ]:
# Feature Importance
# After training, XGBoost can tell us how much each feature contributed
# to making correct predictions across all trees.
#
# The importance score is measured by Gain:
# every time a feature is used to split a node, it reduces the loss function.
# Importance = average gain brought by that feature across all splits and trees.
#
# Higher score → the feature was used more often and more effectively
# Lower score  → the feature contributed little to the predictions
#
# In a medical context this is valuable: it tells us which clinical measurements
# are the strongest predictors of heart disease, which can help clinicians
# prioritise which tests or observations matter most for diagnosis
importances   = best_xgb.feature_importances_
feature_names = X.columns

importance_df = (
    pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    .sort_values('Importance', ascending=False)
    .reset_index(drop=True)
)

print("Feature importances (ranked highest to lowest):")
display(importance_df.round(4))

# Bar chart
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data    = importance_df,
    x       = 'Importance',
    y       = 'Feature',
    palette = 'plasma',       # Different from RF's viridis for visual distinction
    ax      = ax
)
ax.set_title('Feature Importance — XGBoost', fontsize=13)
ax.set_xlabel('Importance Score (Gain)', fontsize=11)
ax.set_ylabel('Feature', fontsize=11)
plt.tight_layout()

# Save plot and CSV
fi_plot_path = os.path.join(PLOTS_DIR, f"{MODEL_NAME}_feature_importance.png")
plt.savefig(fi_plot_path, dpi=150)
plt.show()
print(f"\nPlot saved to : {fi_plot_path}")

fi_csv_path = os.path.join(EVAL_DIR, f"{MODEL_NAME}_feature_importance.csv")
importance_df.to_csv(fi_csv_path, index=False)
print(f"CSV  saved to : {fi_csv_path}")

### Result Interpretation

#### Best Parameters Found
```
colsample_bytree  = 0.7
learning_rate     = 0.1
max_depth         = 3
min_child_weight  = 1
n_estimators      = 100
scale_pos_weight  = 1
Best CV Recall    = 0.9037
```

GridSearchCV searched across 3,456 combinations and selected a model of 100
boosting rounds, each tree capped at depth 3, considering 70% of features per
tree, with a learning rate of 0.1 and equal class weights.

**Why these specific values?**

- **n_estimators = 100:** The search found that 100 boosting rounds was
sufficient for the model to reach its best recall on this dataset. Each round
corrects the errors of the previous one, and with a learning rate of 0.1 the
model converged quickly without needing more rounds.

- **max_depth = 3:** Shallow trees were preferred over deeper ones. A depth of
3 means each tree can make at most 3 consecutive splits before reaching a
final prediction. This keeps individual trees simple and prevents them from
memorising specific patient patterns, the model learns broad, generalisable
rules instead.

- **learning_rate = 0.1:** Each tree contributes 10% of its prediction to the
final answer. This moderate learning rate balances speed of learning with
stability; too high and the model overshoots the optimal solution, too low
and it requires many more trees to converge.

- **colsample_bytree = 0.7:** Each tree only sees 70% of the 16 features
randomly selected at each round. This introduces diversity between trees,
preventing any single feature from dominating all predictions and reducing
the risk of overfitting.

- **min_child_weight = 1:** A leaf node requires a minimum weight sum of 1
before a split is made. The low value here is consistent with the shallow
max_depth, the trees are already constrained by their depth, so aggressive
leaf pruning is not needed.

- **scale_pos_weight = 1:** Equal weights produced better recall than the
class ratio alternative. This confirms that the 55/45 class distribution
(507 disease vs 410 healthy) is balanced enough that artificial reweighting
introduces unnecessary bias rather than correcting a real imbalance problem.

---

#### Cross-Validation Results

| Metric | Train Mean | Validation Mean | Gap | Std Dev |
|---|---|---|---|---|
| Accuracy | 0.9335 | 0.8677 | 0.0658 | 0.0228 |
| Precision | 0.9326 | 0.8675 | 0.0651 | 0.0481 |
| Recall | 0.9481 | 0.9037 | 0.0444 | 0.0275 |
| F1-Score | 0.9403 | 0.8836 | 0.0567 | 0.0145 |
| ROC-AUC | 0.9840 | 0.9292 | 0.0548 | 0.0136 |

**What the Gap tells us — overfitting assessment:**
All gaps fall between 0.044 and 0.066, well within the acceptable Gap. This means the model performs only slightly better on data it was
trained on compared to data it has never seen, a sign of healthy
generalisation. The most important gap is Recall at 0.0444, which is the
lowest of all metrics and confirms the model's ability to detect heart disease
cases transfers well to unseen patients. The mild gaps observed in Accuracy
and Precision are a natural consequence of XGBoost's boosting mechanism,
which fits training data more tightly than simpler models, but the small
magnitude confirms this does not constitute harmful overfitting
(Cawley & Talbot, 2010).

**What the Std Dev tells us — stability assessment:**
ROC-AUC has the lowest std dev of 0.0136, meaning the model's ability to
separate the two classes is highly consistent across all 5 folds. F1-Score
follows closely at 0.0145. Recall at 0.0275 is slightly higher but still
acceptable, this means the model detects heart disease cases at a consistent
rate regardless of which patients make up the validation fold. This stability
is critical in a medical context where inconsistent performance across
different patient groups would undermine clinical trust.

**Why Recall is the most clinically important metric:**
A Validation Recall of 0.9037 means the model catches 90% of actual heart
disease cases on average across the 5 training folds. The remaining 10% are
false negatives, patients the model incorrectly classifies as healthy. In a
medical screening system, these are the most dangerous errors because a missed
diagnosis means a sick patient leaves without treatment or follow-up care.
This is precisely why the model was optimised for recall rather than accuracy
or ROC-AUC during the GridSearchCV search.

---

#### Test Set Results

| Accuracy | Precision | Recall | F1-Score | ROC-AUC |
|---|---|---|---|---|
| 0.8370 | 0.8000 | 0.9412 | 0.8649 | 0.9285 |

**Accuracy 83.70%:** The model correctly classified 154 out of 184 completely
unseen patients. 30 patients were misclassified (24 false positives and 6
false negatives). For a dataset of 917 samples and 16 features, this is a
realistic and honest result that reflects genuine learning.

**Precision 80.00%:** When the model predicts Heart Disease, it is correct
80% of the time. 24 out of every 120 patients flagged as sick are actually
healthy. This is the deliberate trade-off of lowering the decision threshold
to 0.199, we accept more false alarms in exchange for catching more true
disease cases.

**Recall 94.12%:** The model caught 96 out of 102 actual heart disease
patients, missing only 6. A recall of 94.12% means roughly 1 in 17 sick
patients would not be flagged. This is the primary goal of the model and
represents a strong result, in a medical screening context, catching 96 out
of 102 disease cases significantly reduces the risk of missed diagnoses.

**F1-Score 0.8649:** The harmonic mean of Precision and Recall reflects the
balance between the two. The score of 0.8649 confirms that while the model
prioritises recall, it does not sacrifice precision entirely — it maintains a
reasonable balance between catching disease cases and minimising false alarms.

**ROC-AUC 0.9285:** This means that if you randomly selected one heart disease
patient and one healthy patient from the test set, the model would correctly
assign a higher risk score to the sick patient 92.85% of the time across every
possible decision threshold. An AUC above 0.90 is considered excellent in
medical AI literature and confirms the model has genuinely learned to separate
the two classes rather than memorising patterns from the training data.

**CV vs Test agreement — representativeness check:**

| Metric | CV Validation | Test Set | Difference |
|---|---|---|---|
| Accuracy | 0.8677 | 0.8370 | 0.031 |
| Precision | 0.8675 | 0.8000 | 0.068 |
| Recall | 0.9037 | 0.9412 | +0.038 |
| F1-Score | 0.8836 | 0.8649 | 0.019 |
| ROC-AUC | 0.9292 | 0.9285 | 0.001 |

All differences are small, the largest is Precision at 0.068. Notably,
Recall on the test set (0.9412) is actually higher than the CV validation
mean (0.9037), which confirms the model generalises well and did not benefit
from an unusually easy test set. The near-perfect agreement on ROC-AUC
(difference of only 0.001) is particularly strong evidence that there is no
data leakage and the evaluation is fully unbiased.

---

#### Confusion Matrix
```
                      Predicted No Disease    Predicted Heart Disease
Actual No Disease          TN = 58                  FP = 24
Actual Heart Disease       FN = 6                   TP = 96
```

**True Negatives (58):** The model correctly identified 58 out of 82 healthy
patients (70.7% of healthy patients were correctly cleared). The remaining
24 were flagged as disease, the deliberate cost of lowering the threshold
to maximise recall.

**False Positives (24):** 24 healthy patients were incorrectly flagged as
having heart disease (29.3% false alarm rate among healthy patients). These
patients would undergo unnecessary further testing, causing anxiety and
additional healthcare cost. However, in a screening context this is the more
acceptable error type compared to a missed diagnosis, further clinical
testing can rule out disease, but a missed diagnosis receives no follow-up
at all.

**False Negatives (6):** Only 6 patients who actually had heart disease were
classified as healthy (5.9% missed diagnosis rate among sick patients). These
are the most dangerous errors, each represents a patient who could leave
without receiving necessary treatment or follow-up. Reducing this number from
the default threshold result of 15 missed cases down to just 6 is the direct
result of threshold optimisation, and represents the core clinical achievement
of this model.

**True Positives (96):** The model correctly identified 96 out of 102 heart
disease patients (94.1% detection rate, consistent with the recall metric
reported above).

---

#### Feature Importance

| Rank | Feature | Importance |
|------|---------|------------|
| 1 | ST_Slope_Flat | 0.2388 |
| 2 | ST_Slope_Up | 0.2317 |
| 3 | ExerciseAngina | 0.1819 |
| 4 | Oldpeak | 0.0466 |
| 5 | Sex | 0.0466 |
| ... | ... | ... |
| 13–16 | RestingECG, Chol_category | < 0.020 each |

The top 5 features together drive **65.56%** of XGBoost's predictions, meaning
just 5 out of 16 features are responsible for nearly two thirds of every decision
the model makes. The three features at the top of this ranking (ST_Slope_Flat,
ST_Slope_Up, and ExerciseAngina) all come from the same clinical procedure: the
exercise stress test. A model with no prior medical knowledge reproducing the same
diagnostic priorities that cardiologists rely on is a meaningful indicator that the
patterns it learned are grounded in reality.

### ST_Slope_Flat & ST_Slope_Up — Ranks 1 & 2 (Combined 47.05%)

As established in the EDA, the ST segment on an ECG changes shape depending on how well the
heart handles physical exertion. A flat segment is a warning sign, indicating the heart is not
responding normally to the increased demand. An upward slope is what a healthy heart produces
under the same conditions. This distinction was confirmed by Lauer et al. (1996) as one of the
strongest non-invasive signals for detecting heart disease, and XGBoost arrived at the same
conclusion without any clinical guidance.

These two categories account for **854 of the 917 patients (93.1%)** in the dataset.
**82.8% of Flat patients carry a heart disease diagnosis**, while **80.3% of Up patients do not**.
The scale of this separation, two groups pointing in opposite directions with over 80% certainty
each, is what drives their combined **47.05% share of total feature importance**.


> Lauer, M.S., et al. (1996). *Circulation*, 93(8), 1520–1526.
> https://doi.org/10.1161/01.cir.93.8.1520

### ExerciseAngina — Rank 3 (18.19%)

As established in the EDA, when the coronary arteries are narrowed, the heart struggles to keep
up with the oxygen demand that physical activity places on it. One of the clearest signs of this
is chest pain during exercise, a symptom known as exercise-induced angina. Weiner et al. (1978)
found that when chest pain appeared together with abnormal ECG readings during exercise testing,
coronary artery disease was confirmed in 95% of those patients. XGBoost picked up on exactly
this relationship.

Within the dataset, **371 patients (40.5%) experienced chest pain during exercise** while
**546 (59.5%) did not**. The disease rate among those who did was **85.2%**, compared to just
**35.0%** among those who did not. A difference of more than 50 percentage points makes this
feature one of the most decisive in the entire model, accounting for **18.19% of total feature
importance**.

> Weiner, D.A., et al. (1978). *American Heart Journal*, 96(4), 458–462.
> https://doi.org/10.1016/0002-8703(78)90155-2

### Oldpeak — Rank 4 (4.66%)

As established in the EDA, Oldpeak captures how much the ST segment drops during exercise
relative to its resting position. A larger drop reflects a heart that is under greater strain,
and the further it falls, the more the cardiovascular system is being pushed beyond what its
blood supply can support. Miranda et al. (1991) established that this measure is a dependable
indicator of coronary artery disease. As noted in the EDA pair plot, disease cases gather at
Oldpeak values above 1.5, while healthy patients cluster near zero. Among the **100 patients
with Oldpeak exceeding 2.0**, **91 (91%) were diagnosed with heart disease**, a concentration
that explains why this feature contributes **4.66% of total feature importance** even as it
ranks lower here than in the Random Forest model.

> Miranda, C.P., et al. (1991). *American Heart Journal*, 122(6), 1617–1628.
> https://doi.org/10.1016/0002-8703(91)90279-q

### Sex — Rank 5 (4.66%)

As established in the EDA, coronary artery disease tends to develop earlier and more severely
in men than in women. The dataset contains **724 male patients (78.9%)** and **193 female
patients (21.1%)**. Among males, **63.1% have heart disease**, while among females that figure
drops to **25.9%**, a pattern XGBoost detected purely from the data.

That said, the prominence of Sex in the rankings is not purely biological. With males making up
nearly four in every five patients, any signal associated with being male gets reinforced across
the vast majority of predictions. Had the dataset been more evenly split between sexes, this
feature would likely rank lower. What the model learned is accurate for this data, but the
ranking also reflects who is in the dataset, earning Sex **4.66% of total feature importance**.

### Why Bottom Features Ranked Low (RestingECG, Chol_category — < 0.020 each)

As established in the EDA, features measured at rest such as ECG readings and cholesterol
categories carry some signal but cannot compete with what the exercise stress test reveals. By
the time XGBoost has processed ST slope, exercise angina, and Oldpeak, the incremental value
of knowing a patient's resting ECG pattern or cholesterol band is minimal. The heart's behaviour
under physical demand is a far more direct window into its condition than any measurement taken
while it is at rest.

### Overall Clinical Validity

The three most important features (ST_Slope_Flat, ST_Slope_Up, and ExerciseAngina) are all
derived from the exercise stress test, the cornerstone non-invasive procedure in cardiology for
evaluating heart disease risk. These are precisely the readings that cardiologists focus on when
interpreting stress test results. XGBoost, given nothing but patient records and outcomes,
independently converged on the same diagnostic framework. That alignment between a data-driven
model and decades of clinical evidence is a strong signal that the model captured genuine
medical patterns rather than artefacts of the training data.

# Extra Trees

## Section 1. Rationale for Choosing Extra Trees:

Extra Trees (Extremely Randomized Trees) is a tree-based learning algorithm that is widely used in machine learning for both classification and regression tasks. It is built upon decision trees, which makes it well-suited for capturing complex patterns in data, especially non-linear relationships. Extra Trees works by constructing multiple decision trees in parallel and combining their outputs, making it a powerful multi-tree model. Unlike a single decision tree, it introduces additional randomness during splitting, which helps improve generalisation and reduce overfitting on structured tabular data. (Geurts et al., 2006).

### **Why Extra Trees is a Good Fit for Our Data?**

#### i. Dataset Characteristics (Size, Feature Types, and Linearity)

- Our dataset is a small structured tabular medical dataset with a mix of continuous, binary, and one-hot encoded categorical features. This makes Extra Trees a suitable choice because it works especially well on tabular data and does not require assumptions about linearity between the predictors and the target. In our case, features such as Oldpeak, MaxHR, and the encoded categorical variables are likely to interact in non-linear ways, and Extra Trees can capture these interactions naturally through repeated tree splits. (GeeksforGeeks, 2025).

- Another reason Extra Trees fits this dataset well is its high level of randomisation. Unlike a single decision tree, which can become too sensitive to the training data, Extra Trees builds many trees and introduces additional randomness when choosing split points. This is useful for our dataset size because it helps reduce overfitting while still allowing the model to learn meaningful patterns from the data. (Geurts et al., 2006).

- Extra Trees is also appropriate because some of our features contain extreme values and different value ranges. Since tree-based methods rely on threshold-based splitting rather than distance calculations, they are generally less affected by outliers and do not require feature scaling to perform effectively. (GeeksforGeeks, 2025).

#### ii. Problem Type (Binary Classification)

The task in this project is to predict whether a patient has heart disease or not, which makes it a binary classification problem.

Extra Trees is well suited for binary classification tasks because:

- It directly supports classification problems by building multiple randomized decision trees and combining their predictions through majority voting (Geurts et al., 2006).
- The algorithm reduces variance by introducing additional randomness when selecting split thresholds, which helps improve generalisation and stability when predicting class labels. (Berrouachedi et al., 2024).
- Tree-based classifiers like Extra Trees perform particularly well on structured tabular datasets, where relationships between features and the target variable may be complex and non-linear. (Berrouachedi et al., 2024).

Since the task involves separating patients into two classes (Heart Disease vs No Disease) using structured medical attributes, the Extra Trees classifier is an appropriate model for this prediction problem.

#### iii. Model Strengths and Weaknesses for Our Specific Problem

**Strengths:**

- Handles non-linear relationships well, which is important in medical data where predictors do not affect the target in a simple linear way. (GeeksforGeeks, 2025).
- Works effectively on structured tabular datasets with mixed feature types such as continuous, binary, and one-hot encoded variables. (Berrouachedi et al., 2024).
- Introduces more randomness than a standard decision tree, which helps reduce overfitting and improve generalisation. (Geurts et al., 2006).

**Weaknesses:**

- Can still overfit if the trees are not properly controlled through hyperparameter tuning. (GeeksforGeeks, 2025).
- The added randomness, can make the performance vary slightly depending on the chosen hyperparameters and random state. (GeeksforGeeks, 2025).

Overall, Extra Trees is a strong choice for our problem because it can model complex patterns in structured medical data while maintaining good generalisation performance on unseen patients.

---

> **References:**
>
> Geurts, P., Ernst, D., & Wehenkel, L. (2006). *Extremely randomized trees*.
Machine Learning, 63, 3–42. https://doi.org/10.1007/s10994-006-6226-1
>
> GeeksforGeeks. (2025, July 12). *ML | Extra Tree Classifier for Feature Selection*. GeeksforGeeks. https://www.geeksforgeeks.org/machine-learning/ml-extra-tree-classifier-for-feature-selection/
>
> Berrouachedi, A., Jaziri, R., & Bernard, G. (2024). *Enhancing Multi-Label Classification Through Deep Extra-Trees and Transformation Techniques*. 2024 IEEE/ACS 21st International Conference on Computer Systems and Applications (AICCSA), 1–6. https://ieeexplore.ieee.org/document/10912617
>
> GeeksforGeeks. (2025, July 23). *RandomForestClassifier vs ExtraTreesClassifier in scikit learn*. GeeksforGeeks. https://www.geeksforgeeks.org/machine-learning/randomforestclassifier-vs-extratreesclassifier-in-scikit-learn/

## Section 2. Implementation & Training

### Imports & Setup

In [ ]:
# Standard libraries
import pickle
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
warnings.filterwarnings('ignore')

# Model
from sklearn.ensemble import ExtraTreesClassifier

# Splitting & cross-validation
from sklearn.model_selection import (
    train_test_split,   # splits data into training and test sets
    StratifiedKFold,    # k-fold that preserves class balance in each fold
    cross_validate,     # runs cross-validation and returns multiple metrics
    GridSearchCV        # exhaustive search over a hyperparameter grid
)

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,     # proportion of correct predictions
    precision_score,    # of all predicted positives, how many are truly positive
    recall_score,       # of all actual positives, how many were correctly predicted
    f1_score,           # harmonic mean of precision and recall
    confusion_matrix,   # table showing TP, TN, FP, FN counts
    roc_auc_score,      # area under the ROC curve
    roc_curve           # false positive rate vs true positive rate at all thresholds
)

# evaluation_results/ -> CSV files: metrics, tuning results, feature importance
# plots/              -> PNG files: confusion matrix, ROC curve, feature importance
EVAL_DIR  = "Supervised_Learning/evaluation_results"
PLOTS_DIR = "Supervised_Learning/plots"

os.makedirs(EVAL_DIR,  exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# Model name prefix
# Used in every saved file name so outputs are clearly labelled per model
MODEL_NAME = "extra_trees"

print("Libraries imported successfully.")
print(f"Evaluation results → {EVAL_DIR}")
print(f"Plots              → {PLOTS_DIR}")

### Data Loading

In [ ]:
# Load the preprocessed dataset
# This CSV was produced by the EDA notebook — it contains cleaned, encoded,
# and normalised features ready for modeling. No further preprocessing is needed.
DATA_PATH = "Dataset/preprocessed_heart_data.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

### Feature & Target Split

In [ ]:
# Separate input features from the target variable
# X -> all feature columns (what the model uses to make predictions)
# y -> the target column   (what the model is trying to predict)
TARGET = "HeartDisease"   # 0 = No Disease, 1 = Heart Disease

X = df.drop(columns=[TARGET])
y = df[TARGET]

print(f"Features (X): {X.shape[1]} columns")
print(f"\nTarget (y) distribution:")
print(y.value_counts().rename({0: 'No Disease (0)', 1: 'Heart Disease (1)'}))

### Train/Test Split

In [ ]:
# Create or load the 80/20 train/test split
# This cell implements the first and most critical step of our methodology:
# locking away 20% of the data before any training or tuning begins.
#
# IMPORTANT: This split is created ONCE and shared across all models.
# If each model used a different split, performance differences could be due
# to the split rather than the model — making comparison unfair.
# Saving and loading the same indices guarantees a completely fair comparison.
SPLIT_PATH = "split_indices.pkl"

if not os.path.exists(SPLIT_PATH):
    # stratify=y   → preserves the class ratio in both sets
    # random_state → fixes the random seed so the split is reproducible
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    split_indices = {
        "train_indices": X_train.index.tolist(),
        "test_indices" : X_test.index.tolist()
    }
    with open(SPLIT_PATH, "wb") as f:
        pickle.dump(split_indices, f)

    print("Split created and saved to split_indices.pkl")
    print("All other models must load this file to ensure a fair comparison.")
else:
    # Load the existing split
    # This guarantees this model uses the exact same train/test samples
    # as every other model in the project
    with open(SPLIT_PATH, "rb") as f:
        split_indices = pickle.load(f)

    X_train = X.iloc[split_indices["train_indices"]]
    X_test  = X.iloc[split_indices["test_indices"]]
    y_train = y.iloc[split_indices["train_indices"]]
    y_test  = y.iloc[split_indices["test_indices"]]

    print("Existing split loaded from split_indices.pkl")
    print("This Extra Tress model uses the EXACT same train/test samples as XGBoost and Random Forest.")

print(f"\nTraining set : {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test set     : {X_test.shape[0]}  samples ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\nClass distribution in training set:")
print(y_train.value_counts().rename({0: 'No Disease', 1: 'Heart Disease'}))
print(f"\nClass distribution in test set:")
print(y_test.value_counts().rename({0: 'No Disease', 1: 'Heart Disease'}))

### Hyperparameter Tuning Process

#### Extra Trees Hyperparameter Selection

Extra Trees has a set of hyperparameters that directly controls the construction and behaviour of trees during training. These settings influence multiple parts of the model, including how many trees contribute to the final prediction, how complex each tree is allowed to become, how aggressively the data is split, and how much randomness is introduced into the learning process. Since these hyperparameters interact with one another, choosing effective values requires structured tuning based on the dataset.

#### Why These Specific Hyperparameters Were Selected for Tuning
ExtraTreesClassifier has multiple configurable parameters in total. We selected 5 to tune, specifically the ones that most directly control the main sources of error in Extra Trees: overfitting, underfitting, and instability across trees. The remaining parameters were intentionally left at their defaults because they either address similar effects indirectly, require more specialised justification to tune meaningfully, or mainly affect implementation behaviour rather than predictive performance.

**Tuned parameters:**

| Parameter           | Why Tuned                                                                                                |
|---------------------|----------------------------------------------------------------------------------------------------------|
| `n_estimators`      | Number of trees in the ensemble — To control how many trees contribute to the final prediction and improve stability   |
| `max_depth`         | Maximum depth of each tree — To control tree complexity and prevent overfitting or underfitting            |
| `min_samples_split` | Minimum number of samples required to split an internal node — To prevent splits based on very small groups of samples |
| `min_samples_leaf`  | Minimum number of samples required in a leaf — To avoid overly specific leaves and improve generalisation                      |
| `max_features`      | How many features to consider at each split — To control randomness at each split and increase diversity across trees                                    |

The remaining 14 parameters were excluded because they either overlap with the behavior already controlled by the tuned parameters, are mainly used for pruning or additional constraints beyond the scope of this work (`min_weight_fraction_leaf`, `max_leaf_nodes`, `min_impurity_decrease`, `ccp_alpha`, `monotonic_cst`), are fundamental design choices that were not intended to be changed (`criterion`, `bootstrap`), serve diagnostic or incremental-training purposes rather than improving predictive quality (`oob_score`, `warm_start`), or are infrastructure and execution settings that affect reproducibility or computation rather than model performance (`n_jobs`, `random_state`, `verbose`, `max_samples`).


> **Reference:** scikit-learn developers. (n.d.). *sklearn.ensemble.ExtraTreesClassifier*.
> scikit-learn. https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.ExtraTreesClassifier.html

In [ ]:
# Step 1: Define the hyperparameter search grid
# param_grid is a menu of candidate settings for GridSearchCV to test.
# GridSearchCV will test every possible combination using 5-fold cross-validation,
# 4×3×4×4×3 = 576 combinations × 5 folds = 2,880 model fits in total.
# then select the combination with the highest average recall score.
param_grid = {
    # Number of trees built
    # More trees usually improve stability, but after a certain point the gain becomes small
    'n_estimators': [200, 300, 400, 500],

    # Maximum tree depth
    # Controls how complex each tree is allowed to become
    # We avoid very shallow trees (which may underfit) and very deep trees (which may overfit)
    # This range tests moderate depths that are realistic for a small tabular medical dataset such as ours
    'max_depth': [5, 8, 10],

    # Minimum samples required to split an internal node
    # Higher values make the trees more conservative and reduce overfitting
    # These values were chosen to stop the model from creating splits based on tiny patient groups
    'min_samples_split': [10, 15, 20, 25],

    # Minimum samples required in a leaf node
    # Prevents the model from ending with leaves that represent only a few patients
    # This is especially important in medical prediction, where overly specific rules are risky
    'min_samples_leaf': [2, 3, 4, 6],

    # Number of features considered at each split
    # 'sqrt' and 'log2' are standard strong choices for tree ensembles
    # None means all features are considered, which can improve fit but may reduce diversity
    'max_features': ['sqrt', 'log2', None],
}

total_combinations = np.prod([len(v) for v in param_grid.values()])
print(f"Total combinations : {total_combinations}")
print(f"Total fits         : {total_combinations} × 5 folds = {total_combinations*5}")

#### GridSearchCV & Model Training

In [ ]:
# Step 2: Define the cross-validation strategy
# This tells GridSearchCV HOW to evaluate each combination fairly:
# — Split the training samples into 5 equal folds
# — Each fold takes a turn being the validation set exactly once
# — Stratified: every fold preserves the class ratio of the full training set
# — shuffle + random_state=42: same fold assignments every run → reproducible results
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Step 3: Run the exhaustive search
# GridSearchCV ties all three components together:
#   estimator  → the model to train (ExtraTreesClassifier)
#   param_grid → the combinations to try (576 total)
#   cv         → how to evaluate each combination (5-fold stratified CV)
#   scoring    → what metric to optimise (ROC-AUC)
#
# Why ROC-AUC and not Recall, F1-score, or Accuracy?
# Recall focuses only on how many true Heart Disease cases are caught at one
# fixed threshold, while F1-score balances Precision and Recall only at that
# same threshold. Accuracy is also threshold-dependent and can look strong even
# when the model makes an unhelpful trade-off between False Positives and False
# Negatives.
#
# ROC-AUC is the most suitable choice here because it evaluates how well the
# model separates Heart Disease patients from healthy patients across ALL
# possible decision thresholds, not just the default 0.5 cutoff. This gives a
# stronger measure of the model's underlying discrimination ability. In a
# medical setting, this is especially valuable because the final threshold can
# later be adjusted clinically to prioritise Recall without retraining the model.
#
# What happens when .fit() is called:
# 1. For each of 576 combinations → train on 4 folds, validate on 1 fold → 5 times
# 2. Average the 5 ROC-AUC scores for each combination
# 3. Pick the combination with the highest average score
# 4. Retrain that winning combination on ALL 733 training samples from scratch
# 5. Store the final model as grid_search.best_estimator_
# n_jobs=-1 → uses all available CPU cores to run fits in parallel
grid_search = GridSearchCV(
    estimator=ExtraTreesClassifier(random_state=42),
    param_grid=param_grid,            # Full grid → all 576 combinations tested
    cv=cv_strategy,                   # 5-fold stratified CV per combination
    scoring='roc_auc',                # Optimise discrimination across all thresholds
    n_jobs=-1,                        # Use all CPU cores in parallel
    verbose=1                         # Print progress during search
)

print(f"Starting Extra Trees GridSearchCV ({total_combinations} combinations × 5 folds = {total_combinations*5} fits)...")
print("Optimising for: ROC-AUC — strongest class separation across thresholds\n")

grid_search.fit(X_train, y_train)

print("\nSearch complete.")
print(f"Best parameters : {grid_search.best_params_}")
print(f"Best CV ROC-AUC : {grid_search.best_score_:.4f}")

#### Tuning Results & Best Model Extraction

In [ ]:
# Save and display the full hyperparameter search results
# grid_search.cv_results_ contains the score for every combination tried.
# Saving all 576 rows to CSV allows us to review close competitors and
# understand how sensitive performance is to each hyperparameter.
tuning_df   = pd.DataFrame(grid_search.cv_results_)
tuning_path = os.path.join(EVAL_DIR, f"{MODEL_NAME}_hyperparameter_tuning.csv")
tuning_df.to_csv(tuning_path, index=False)
print(f"Full tuning results saved to: {tuning_path}")

# Display the top 10 winning configurations ranked by validation ROC-AUC
# mean_validation_score → average ROC-AUC across 5 validation folds
#                         Note: scikit-learn internally calls this 'mean_test_score'
#                         — the word 'test' here refers to the validation fold,
#                         NOT the held-out test set locked away earlier
# std_validation_score  → consistency of the score across 5 folds
#                         lower std = more reliable combination
# rank_validation_score → 1 = best combination overall
top10 = (
    tuning_df[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']]
    .sort_values('rank_test_score')
    .head(10)
    .round(4)
    .rename(columns={
        'mean_test_score' : 'mean_validation_score',
        'std_test_score'  : 'std_validation_score',
        'rank_test_score' : 'rank_validation_score'
    })
)
print("\nTop 10 hyperparameter configurations:")
display(top10)

# Extract the final trained model
# best_estimator_ is NOT one of the 2,880 temporary models from the search.
# After identifying the winning combination, GridSearchCV automatically trained
# a brand new model from scratch using those settings on ALL training samples.
# This final model has seen more data than any fold-level model during the search,
# making it the strongest possible version of the winning configuration.
#
# All subsequent cells use best_extra_trees — it is the only model we evaluate and report.
best_extra_trees = grid_search.best_estimator_

print("\nBest Extra Trees configuration:")
print(best_extra_trees)

## Section 3. Evaluation Metrics & Visualizations

### 5-Fold Cross-Validation

In [ ]:
# Stage 1 Evaluation: Stability and Overfitting Check
# We now re-evaluate best_extra_trees using 5-fold cross-validation on the training set.
# This is DIFFERENT from what GridSearchCV did internally:
#   — GridSearchCV used CV to COMPARE 576 combinations (selection bias present)
#   — cross_validate uses CV to MEASURE this one fixed model honestly
#
# By collecting both training scores and validation scores, we can compute
# the Gap: the difference between how well the model fits training data
# vs how well it generalises to data it was not trained on.
#
# A small gap means the model generalises well.
# A large gap means the model memorised training data (overfitting).
cv_results = cross_validate(
    best_extra_trees,
    X_train, y_train,
    cv                 = cv_strategy,
    scoring            = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
    return_train_score = True
)

# Build the comparison table
# Train Mean      → average score when evaluated on the data the model was trained on
# Validation Mean → average score when evaluated on the fold held out for validation
# Gap             → Train Mean - Validation Mean
# Std Dev         → how consistent the validation score is across 5 folds
#                   < 0.03 → stable model
#                   > 0.05 → unstable
cv_summary = pd.DataFrame({
    'Metric'          : ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Train Mean'      : [
        cv_results['train_accuracy'].mean(),
        cv_results['train_precision'].mean(),
        cv_results['train_recall'].mean(),
        cv_results['train_f1'].mean(),
        cv_results['train_roc_auc'].mean()
    ],
    'Validation Mean' : [
        cv_results['test_accuracy'].mean(),
        cv_results['test_precision'].mean(),
        cv_results['test_recall'].mean(),
        cv_results['test_f1'].mean(),
        cv_results['test_roc_auc'].mean()    # Primary tuning metric for this model
    ],
    'Gap'             : [
        cv_results['train_accuracy'].mean()  - cv_results['test_accuracy'].mean(),
        cv_results['train_precision'].mean() - cv_results['test_precision'].mean(),
        cv_results['train_recall'].mean()    - cv_results['test_recall'].mean(),
        cv_results['train_f1'].mean()        - cv_results['test_f1'].mean(),
        cv_results['train_roc_auc'].mean()   - cv_results['test_roc_auc'].mean()
    ],
    'Std Dev'         : [
        cv_results['test_accuracy'].std(),
        cv_results['test_precision'].std(),
        cv_results['test_recall'].std(),
        cv_results['test_f1'].std(),
        cv_results['test_roc_auc'].std()
    ]
}).round(4)

print("5-Fold Cross-Validation Results (Training vs Validation):")
display(cv_summary)

# Save to CSV
cv_path = os.path.join(EVAL_DIR, f"{MODEL_NAME}_cv_results.csv")
cv_summary.to_csv(cv_path, index=False)
print(f"\nSaved to: {cv_path}")

### Final Evaluation on the Held-Out Test Set

In [ ]:
# Stage 2 Evaluation: Final Unbiased Test on the Held-Out Test Set
# This is the only stage where the final Extra Trees model is evaluated on X_test.
# These 184 samples were separated before any training or tuning, so they provide
# an unbiased estimate of how the model performs on unseen patients.
#
# predict()       → returns the predicted class label for each patient
# predict_proba() → returns class probabilities
#                   [:, 1] extracts the probability of Heart Disease (class 1),
#                   which is required for ROC-AUC

y_pred = best_extra_trees.predict(X_test)
y_prob = best_extra_trees.predict_proba(X_test)[:, 1]

# Calculate the final evaluation metrics on the held-out test set
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_prob)

print("EXTRA TREES — FINAL TEST SET RESULTS")

results_df = pd.DataFrame([{
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'ROC-AUC': roc_auc
}]).round(4)

display(results_df)

# Save results
metrics_path = os.path.join(EVAL_DIR, f"{MODEL_NAME}_metrics.csv")
results_df.to_csv(metrics_path, index=False)
print(f"Results saved to: {metrics_path}")

### Confusion Matrix

The confusion matrix below summarises how the Extra Trees model performed on all 184 unseen test patients. It shows where the model was correct and where it made mistakes by comparing each patient’s true condition with the model’s predicted class. The rows represent the actual class, while the columns represent the predicted class.

In [ ]:
# Confusion Matrix
# A confusion matrix breaks down all test predictions into 4 categories,
# showing exactly WHERE the model is correct and where it makes mistakes.
#
# Reading the matrix:
#   Rows    = the actual (true) label of the patient
#   Columns = what the model predicted
#
# The 4 cells:
#   TN (top-left)     → model predicted No Disease, patient actually has No Disease ✓
#   FP (top-right)    → model predicted Heart Disease, patient is actually Healthy
#                       consequence: unnecessary further testing, patient anxiety
#   FN (bottom-left)  → model predicted No Disease, patient actually has Heart Disease
#                       consequence: missed diagnosis (most dangerous error in medical AI)
#   TP (bottom-right) → model predicted Heart Disease, patient actually has Heart Disease ✓
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=['No Disease (0)', 'Heart Disease (1)'],
    yticklabels=['No Disease (0)', 'Heart Disease (1)'],
    ax=ax
)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label',      fontsize=12)
ax.set_title('Confusion Matrix — Extra Trees', fontsize=13)
plt.tight_layout()

# Save plot
cm_path = os.path.join(PLOTS_DIR, f"{MODEL_NAME}_confusion_matrix.png")
plt.savefig(cm_path, dpi=150)
plt.show()
print(f"Saved to: {cm_path}")

print(f"\nTrue  Negatives (TN): {tn}  — correctly predicted No Disease")
print(f"False Positives (FP): {fp}  — predicted Heart Disease for a healthy patient")
print(f"False Negatives (FN): {fn}  — predicted No Disease for a patient who actually has Heart Disease")
print(f"True  Positives (TP): {tp}  — correctly predicted Heart Disease")

### ROC Curve

The ROC curve below illustrates how well the Extra Trees model distinguishes between heart disease patients and healthy patients across all possible decision thresholds, not only the default threshold of 0.5. The green curve represents the Extra Trees model, while the grey dashed diagonal represents a random classifier. The closer the curve moves toward the top-left corner, the better the model is at achieving high true positive rates while keeping false positive rates low. The AUC (Area Under the Curve) summarises this performance into a single value, and our model achieved 0.9273, which indicates excellent discrimination between the two classes.


In [ ]:
# ROC Curve
# The ROC curve visualises how the model performs at EVERY possible threshold,
# not just the default 0.5. This is especially important in medical systems
# where the threshold may be adjusted by clinicians based on risk tolerance.
#
# X-axis: False Positive Rate (FPR) → proportion of healthy patients falsely flagged
# Y-axis: True Positive Rate (TPR)  → proportion of sick patients correctly caught
#
# Every point on the curve represents a different threshold:
#   — Low threshold (e.g. 0.2)  → catch almost all disease cases but many false alarms
#   — High threshold (e.g. 0.8) → very few false alarms but many missed cases
#
# The grey dashed diagonal = random guessing (AUC = 0.50)
# The green curve = our model — the further it bows toward the top-left, the better
# AUC = area under the curve:
#   0.50 → no better than random  |  0.70–0.80 → acceptable
#   0.80–0.90 → good              |  0.90+ → excellent
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_score   = roc_auc_score(y_test, y_prob)


fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='seagreen', lw=2,
        label=f'Extra Trees (AUC = {auc_score:.4f})')
ax.plot([0, 1], [0, 1], color='grey', linestyle='--', lw=1,
        label='Random Classifier (AUC = 0.50)')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — Extra Trees', fontsize=13)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()

# Save plot
roc_path = os.path.join(PLOTS_DIR, f"{MODEL_NAME}_roc_curve.png")
plt.savefig(roc_path, dpi=150)
plt.show()
print(f"Saved to: {roc_path}")

### Feature Importance

The chart below ranks all 16 features by how much they contributed to the Extra Trees model’s predictions. In this model, feature importance is measured using Mean Decrease in Impurity (MDI). Each time a feature is used to split a node, it helps reduce impurity in the resulting groups. The total reduction contributed by that feature is accumulated across all trees in the ensemble. A higher importance score means the feature played a stronger role in improving predictions, while a lower score means it had less influence on the model’s decisions.

In [ ]:
# Feature Importance — Extra Trees
# In Extra Trees, feature importance is measured using Mean Decrease in Impurity (MDI).
# Every time a feature is used to split a node, it reduces impurity.
# The total impurity reduction contributed by each feature is accumulated
# across all trees, then normalised into feature importance scores.

importances   = best_extra_trees.feature_importances_
feature_names = X.columns

# Create a ranked table
importance_df = (
    pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    })
    .sort_values('Importance', ascending=False)
    .reset_index(drop=True)
)

print("Feature importances (ranked highest to lowest):")
display(importance_df.round(4))

# Horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=importance_df,
    x='Importance',
    y='Feature',
    palette='viridis',
    ax=ax
)

ax.set_title('Feature Importance — Extra Trees', fontsize=13)
ax.set_xlabel('Mean Decrease in Impurity (Importance Score)', fontsize=11)
ax.set_ylabel('Feature', fontsize=11)
plt.tight_layout()

# Save plot and CSV
fi_plot_path = os.path.join(PLOTS_DIR, f"{MODEL_NAME}_feature_importance.png")
plt.savefig(fi_plot_path, dpi=150)
plt.show()
print(f"Plot saved to: {fi_plot_path}")

fi_csv_path = os.path.join(EVAL_DIR, f"{MODEL_NAME}_feature_importance.csv")
importance_df.to_csv(fi_csv_path, index=False)
print(f"CSV saved to: {fi_csv_path}")

### Result Interpretation

#### Best Parameters Found

```
max_depth          = 10
max_features       = ‘sqrt’
min_samples_leaf   = 3
min_samples_split  = 10
n-estimators       = 300
Best CV ROC-AUC    = 0.9249
```

GridSearchCV searched across 576 combinations and selected an Extra Trees model with 300 trees, each capped at a maximum depth of 10, requiring at least 10 samples to split an internal node and at least 3 samples in each leaf, while considering the square root of the total number of features at each split.

**Why these specific values?**

- **max_depth = 10:** A depth of 10 gave the model enough flexibility to capture important non-linear relationships in the data without making the trees excessively deep. Shallower trees were more restrictive, while deeper trees would have increased the risk of overfitting by learning patterns that are too specific to the training set.

- **max_features = ‘sqrt’:** At each split, the model considers only the square root of the total number of features rather than all of them. This introduces useful randomness across trees, increases diversity in the ensemble, and helps reduce overfitting while still allowing the model to learn strong predictive patterns.

- **min_samples_leaf = 3:** Each leaf must contain at least 3 samples, which prevents the model from ending with leaves that represent only one or two unusual patients. This improves the stability of the final predictions and reduces the chance of memorising rare cases instead of learning general patterns.

- **min_samples_split = 10:** Requiring at least 10 samples before a node can split makes the model more conservative. This prevents the trees from creating branches based on very small patient subgroups, which helps improve generalisation and reduces sensitivity to noise.

- **n_estimators = 300:** The search found that 300 trees gave the model the best cross-validated ROC-AUC on this dataset. This means the ensemble benefited from having enough trees to stabilise its predictions, while adding more trees beyond this point did not provide a meaningful improvement.

---

#### Cross-Validation Results

| Metric | Train Mean | Validation Mean | Gap | Std Dev |
|---|---|---|---|---|
| Accuracy | 0.8970 | 0.8541 | 0.0429 | 0.0375 |
| Precision | 0.8891 | 0.8569 | 0.0322 | 0.0535 |
| Recall | 0.9296 | 0.8889 | 0.0407 | 0.0221 |
| F1-Score | 0.9089 | 0.8715 | 0.0374 | 0.0287 |
| ROC-AUC | 0.9729 | 0.9249 | 0.0480 | 0.0219 |

**What the Gap tells us — overfitting assessment:**
All gaps fall between 0.0322 and 0.0480, remaining below the acceptable Gap. This indicates that the Extra Trees model performs only moderately better on the training data than on unseen validation data, which is a healthy sign of generalisation rather than memorisation. The model is learning patterns that transfer to new patients instead of simply fitting noise in the training set. The largest gap appears in ROC-AUC (0.0480), while the smallest appears in Precision (0.0322), showing that the model remains well-controlled even though it is a flexible tree-based ensemble.

**What the Std Dev tells us — stability assessment:**
The validation standard deviations range from 0.0219 to 0.0535, which suggests generally stable performance across the 5 folds. The most stable metric is ROC-AUC (0.0219), meaning the model’s ability to separate heart disease cases from healthy cases is highly consistent regardless of which fold is used for validation. Recall (0.0221) is also very stable, which is especially important in a medical setting because it shows that the model detects disease cases at a reliable rate across different subsets of patients. Precision (0.0535) is the least stable metric, but it is still acceptable for a dataset of this size, where relatively small changes in the number of correctly predicted positives can noticeably affect the score.

**Why Recall is the most clinically important metric:**
A validation Recall of 0.8889 means the model correctly identifies about 88.89% of actual heart disease cases on average across the 5 validation folds. The remaining 11.11% are false negatives, meaning patients who truly have heart disease but are incorrectly classified as healthy. In a medical setting, these are the most dangerous errors because a missed diagnosis can delay treatment or prevent the patient from receiving further clinical attention. For this reason, even though ROC-AUC was used during tuning to measure overall class separation, Recall remains the most clinically important metric because it directly reflects how well the model succeeds in catching real disease cases.

---

#### Test Set Results

| Accuracy | Precision | Recall | F1-Score | ROC-AUC |
|---|---|---|---|---|
| 0.8804 | 0.8704 | 0.9216 | 0.8952 | 0.9273 |

**Accuracy 88.04%:** The model correctly classified 162 out of 184 completely unseen patients. Only 22 patients were misclassified (14 false positives and 8 false negatives). For a dataset of 917 samples and 16 features, this is a strong and realistic result that reflects genuine learning rather than memorisation.

**Precision 87.04%:** When the model predicts Heart Disease, it is correct 87.04% of the time. In practical terms, 14 out of every 108 patients flagged as sick are actually healthy. This means the model keeps false alarms relatively controlled while still remaining sensitive to true disease cases.

**Recall 92.16%:** The model correctly identified 94 out of 102 actual heart disease patients, missing only 8. A recall of 92.16% means roughly 1 in 13 sick patients would not be flagged. In a medical context, this is especially important because recall directly reflects how well the model avoids missed diagnoses, which are the most dangerous type of error.

**F1-Score 89.52%:** The harmonic mean of Precision and Recall shows that the model maintains a strong balance between the two. A score of 0.8952 confirms that the model is not only good at catching real disease cases, but also reasonably controlled in how often it raises false alarms.

**ROC-AUC 92.73%:** This means that if one heart disease patient and one healthy patient were chosen at random from the test set, the model would assign a higher risk score to the sick patient 92.73% of the time across all possible decision thresholds. An AUC above 0.90 is considered excellent and confirms that the Extra Trees model has learned strong class separation rather than simply fitting the training data.

**CV vs Test agreement — representativeness check:**

| Metric | CV Validation | Test Set | Difference |
|---|---|---|---|
| Accuracy | 0.8541 | 0.8804 | +0.0263 |
| Precision | 0.8569 | 0.8704 | +0.0135 |
| Recall | 0.8889 | 0.9216 | +0.0327 |
| F1-Score | 0.8715 | 0.8952 | +0.0237 |
| ROC-AUC | 0.9249 | 0.9273 | +0.0024 |

All differences between the cross-validation results and the held-out test set are small, and in this case the test set scores are actually slightly higher across all five metrics. This is a strong sign that the Extra Trees model generalises well and that its performance is not limited to the folds seen during validation. The closest agreement appears in ROC-AUC, where the difference is only +0.0024, confirming that the model’s ability to separate heart disease patients from healthy patients remains highly consistent on unseen data. The increases in Accuracy, Precision, Recall, and F1-score also suggest that the model maintained stable performance rather than collapsing outside cross-validation.

From a medical perspective, the most important result is the improvement in Recall from 0.8889 in cross-validation to 0.9216 on the test set. This means the model detected an even larger proportion of actual heart disease cases on completely unseen patients, which strengthens confidence in its usefulness as a clinical decision-support tool.

---

#### Confusion Matrix
```
                      Predicted No Disease    Predicted Heart Disease
Actual No Disease          TN = 68                  FP = 14
Actual Heart Disease       FN = 8                   TP = 94
```

**True Negatives (68):** The model correctly identified 68 out of 82 healthy patients, meaning 82.9% of healthy patients were correctly classified as having No Disease. This shows the model is reasonably effective at clearing healthy patients without raising unnecessary concern.

**False Positives (14):** 14 healthy patients were incorrectly flagged as having heart disease, which corresponds to a 17.1% false alarm rate among healthy patients. These patients would likely undergo additional follow-up testing or evaluation. While not ideal, this type of error is generally less dangerous than missing a true disease case.

**False Negatives (8):** 8 patients who actually had heart disease were incorrectly classified as healthy, giving a 7.8% missed diagnosis rate among sick patients. These are the most serious errors in a medical setting because they represent patients who may leave without receiving timely attention or further investigation.

**True Positives (94):** The model correctly identified 94 out of 102 heart disease patients, which corresponds to a 92.2% detection rate. This is consistent with the reported recall and shows that the model successfully captures the large majority of real disease cases.

---

#### Feature Importance

| Rank  | Feature | Importance   |
|-------|---|--------------|
| 1     | ST_Slope_Up | 0.2976       |
| 2     | ST_Slope_Flat | 0.1930       |
| 3     | ExerciseAngina | 0.1438       |
| 4     | ChestPainType_ATA | 0.0670       |
| 5     | Sex | 0.0609       |
| ...   | ... | ...          |
| 14–16 | Chol_category_High, ChestPainType_TA, RestingECG_ST | < 0.011 each |

The top 5 features account for **76.23%** of the model’s total predictive power, which means more than three quarters of the Extra Trees model’s decisions are driven by just five variables.

**ST_Slope_Up (0.2976) and ST_Slope_Flat (0.1930) — combined 49.06%**

As shown in the EDA, the ST segment on an ECG changes shape depending on how well the heart responds to physical exertion. A flat ST slope is a warning sign that the heart is not responding normally to increased demand, while an upward ST slope is the pattern more commonly associated with a healthy cardiac response during exercise. This is why these two features became the strongest predictors in the model: they capture one of the clearest physiological differences between disease and no-disease cases, and they align closely with how clinicians interpret stress-test results in real practice. (Lauer et al., 1996).

The numbers make that separation very clear. These two categories account for 854 of the 917 patients (93.1%) in the dataset. Among patients with ST_Slope_Flat, 82.8% have heart disease. Among patients with ST_Slope_Up, 80.3% do not have heart disease. This means the model sees two very large groups pointing in opposite directions, each with more than 80% certainty, which explains why they contribute such a large combined share of feature importance.

> Lauer, M.S., et al. (1996). *Impaired heart rate response to graded
> exercise.* Circulation, 93(8), 1520–1526.
> https://doi.org/10.1161/01.cir.93.8.1520

**ExerciseAngina (0.1438) — 14.38%**

Exercise-induced angina reflects whether chest pain appears during physical exertion, which is one of the clearest warning signs that the heart may not be receiving enough blood under stress. In the EDA, this feature shows a strong difference between the two classes, so the model naturally learned that it is highly useful for separating disease from no-disease cases. (Weiner et al., 1978).

The numbers support that. Across the full dataset of 917 patients, those with ExerciseAngina show 85.2% heart disease, while those without it show only 35.0% heart disease. That is a very large gap across two groups that together cover the entire dataset, which explains why this feature ranks third and contributes 14.38% of total importance.

> Weiner, D.A., et al. (1978). *The predictive value of anginal chest pain
> as an indicator of coronary disease during exercise testing.* American
> Heart Journal, 96(4), 458–462.
> https://doi.org/10.1016/0002-8703(78)90155-2

**ChestPainType_ATA (0.0670) — 6.70%**

Atypical angina is less classic than typical angina, it still carries useful diagnostic information and may point to underlying coronary disease in a meaningful number of patients. In this dataset, it seems to have provided the model with additional information that helped separate disease from no-disease cases. (Baggiano et al., 2020). The EDA suggests that ChestPainType_ATA provides a strong low-risk pattern for the model. While it is not as dominant as the stress-test features, it still gives the model a useful clinical clue because it is associated much more with the no-disease class than with the disease class.

The numbers make that clear. The ATA category includes 173 of the 917 patients (18.9%) in the dataset. Within that group, 86.1% are in the No Disease class and only 13.9% are in the Disease class. The separation is strong, but because the group itself is smaller than the major ST-slope groups, its total contribution to feature importance is lower at 6.70%.
> Baggiano, A., Guglielmo, M., Muscogiuri, G., Guaricci, A. I., Del Torto, A., & Pontone, G. (2020). Epicardial and microvascular angina or atypical chest pain: Differential diagnoses with cardiovascular magnetic resonance. European Heart Journal Supplements, 22(Suppl E), E42–E47. https://pubmed.ncbi.nlm.nih.gov/32523454/

**Sex (0.0466) — 4.66%**

Men tend to develop the disease earlier and more frequently, while women may show somewhat different symptom patterns. This makes sex a useful predictive feature when combined with other clinical variables, and the model appears to have captured that difference from the data. (Kannel et al., 1976). The EDA shows that Sex captures a strong population-level difference in disease distribution. It is not as directly diagnostic as the stress-test features, but it still gives the model meaningful statistical separation between the classes, which is why it appears in the top five.

The numbers explain that importance. Across the full dataset of 917 patients, 63.1% of males have heart disease, while only 25.9% of females do. On the other side, 74.1% of females fall in the No Disease group. This difference is substantial enough to help the model, even if it is less decisive than the leading clinical variables, which is why it contributes 6.09% of total feature importance.

> Kannel, W.B., Hjortland, M.C., McNamara, P.M., & Gordon, T. (1976).
> *Menopause and risk of cardiovascular disease: the Framingham study.*
> Annals of Internal Medicine, 85(4), 447–452.
> https://doi.org/10.7326/0003-4819-85-4-447

**Bottom features (Chol_category_High, ChestPainType_TA, RestingECG_ST — each < 1.1%)**

These features add only limited predictive value in the Extra Trees model. This suggests that once stronger variables such as ST slope, exercise angina, and chest-pain category are already available, these lower-ranked features contribute very little extra information. Their signal is either weak on its own or already covered by more influential predictors. Moreover, the EDA does not show the same scale of separation for them. Their class distributions are more mixed, so they do not provide the model with strong or clean boundaries between disease and no-disease cases.

---

**Overall clinical validity**

Of the five most important features, the ones most directly related to the exercise stress test are ST_Slope_Up, ST_Slope_Flat, and ExerciseAngina. These features describe how the heart behaves during physical stress: the ST slope reflects changes in the ECG pattern while exercising, and exercise-induced angina captures whether chest pain appears when the heart is under increased demand. These are exactly the kinds of measurements clinicians watch closely during a stress test because they can reveal whether the heart is receiving enough blood during exertion.

The fact that the model gave these features the highest importance means it found them especially useful for separating patients with heart disease from those without it. In other words, when the Extra Trees model built its decision trees, these stress-test-related variables consistently helped create the clearest and most informative splits in the data. That is a strong sign that the model learned medically meaningful patterns: it independently focused on the same stress-related indicators that are already considered important in real clinical interpretation.